In [1]:
# --- Dependency Bootstrap (run this cell first) ---
import shutil
import subprocess
import sys

def _pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "-U"] + args
    print("[deps]", " ".join(cmd))
    subprocess.check_call(cmd)

torch_pkgs = ["torch", "torchvision", "torchaudio"]
runtime_pkgs = [
    "numpy",
    "pillow",
    "tqdm",
    "datasets",
    "pyarrow",
    "pandas",
    "scipy",
    "matplotlib",
]

cuda_index = "https://download.pytorch.org/whl/cu124"
cuda_attempted = False

if shutil.which("nvidia-smi") is not None:
    try:
        cuda_attempted = True
        _pip_install(["--index-url", cuda_index] + torch_pkgs)
        print("[deps] Installed CUDA wheels from cu124 index.")
    except Exception as exc:
        print(f"[deps] CUDA wheel install failed: {exc}")
        print("[deps] Falling back to default PyPI torch wheels.")

if not cuda_attempted:
    _pip_install(torch_pkgs)
elif cuda_attempted:
    # If CUDA install failed, install default wheels as fallback.
    try:
        import torch  # noqa: F401
    except Exception:
        _pip_install(torch_pkgs)

_pip_install(runtime_pkgs)
print("[deps] Dependency install complete. Restart kernel if this is first-time install.")


[deps] /media/fezan/45559F8A629E3B90/tgsc/.venv/bin/python -m pip install -U --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio


Looking in indexes: https://download.pytorch.org/whl/cu124
[deps] Installed CUDA wheels from cu124 index.
[deps] /media/fezan/45559F8A629E3B90/tgsc/.venv/bin/python -m pip install -U numpy pillow tqdm datasets pyarrow pandas scipy matplotlib
[deps] Dependency install complete. Restart kernel if this is first-time install.


In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms


import os
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import AutoAugment, AutoAugmentPolicy, RandomErasing
from tqdm.auto import tqdm

# ResNet

In [3]:
# # from __future__ import absolute_import

# '''Resnet for cifar dataset.
# Ported form
# https://github.com/facebook/fb.resnet.torch
# and
# https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py
# (c) YANG, Wei
# '''

# __all__ = ['resnet']


# def conv3x3(in_planes, out_planes, stride=1):
#     """3x3 convolution with padding"""
#     return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
#                      padding=1, bias=False)


# class BasicBlock(nn.Module):
#     expansion = 1

#     def __init__(self, inplanes, planes, stride=1, downsample=None, is_last=False):
#         super(BasicBlock, self).__init__()
#         self.is_last = is_last
#         self.conv1 = conv3x3(inplanes, planes, stride)
#         self.bn1 = nn.BatchNorm2d(planes)
#         self.relu = nn.ReLU(inplace=True)
#         self.conv2 = conv3x3(planes, planes)
#         self.bn2 = nn.BatchNorm2d(planes)
#         self.downsample = downsample
#         self.stride = stride

#     def forward(self, x):
#         residual = x

#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = self.relu(out)

#         out = self.conv2(out)
#         out = self.bn2(out)

#         if self.downsample is not None:
#             residual = self.downsample(x)

#         out += residual
#         preact = out
#         out = F.relu(out)
#         if self.is_last:
#             return out, preact
#         else:
#             return out


# class Bottleneck(nn.Module):
#     expansion = 4

#     def __init__(self, inplanes, planes, stride=1, downsample=None, is_last=False):
#         super(Bottleneck, self).__init__()
#         self.is_last = is_last
#         self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(planes)
#         self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride,
#                                padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(planes)
#         self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
#         self.bn3 = nn.BatchNorm2d(planes * 4)
#         self.relu = nn.ReLU(inplace=True)
#         self.downsample = downsample
#         self.stride = stride

#     def forward(self, x):
#         residual = x

#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = self.relu(out)

#         out = self.conv2(out)
#         out = self.bn2(out)
#         out = self.relu(out)

#         out = self.conv3(out)
#         out = self.bn3(out)

#         if self.downsample is not None:
#             residual = self.downsample(x)

#         out += residual
#         preact = out
#         out = F.relu(out)
#         if self.is_last:
#             return out, preact
#         else:
#             return out


# class ResNet(nn.Module):

#     def __init__(self, depth, num_filters, block_name='BasicBlock', num_classes=100):
#         super(ResNet, self).__init__()
#         # Model type specifies number of layers for CIFAR-10 model
#         if block_name.lower() == 'basicblock':
#             assert (depth - 2) % 6 == 0, 'When use basicblock, depth should be 6n+2, e.g. 20, 32, 44, 56, 110, 1202'
#             n = (depth - 2) // 6
#             block = BasicBlock
#         elif block_name.lower() == 'bottleneck':
#             assert (depth - 2) % 9 == 0, 'When use bottleneck, depth should be 9n+2, e.g. 20, 29, 47, 56, 110, 1199'
#             n = (depth - 2) // 9
#             block = Bottleneck
#         else:
#             raise ValueError('block_name shoule be Basicblock or Bottleneck')

#         self.inplanes = num_filters[0]
#         self.conv1 = nn.Conv2d(3, num_filters[0], kernel_size=3, padding=1,
#                                bias=False)
#         self.bn1 = nn.BatchNorm2d(num_filters[0])
#         self.relu = nn.ReLU(inplace=True)
#         self.layer1 = self._make_layer(block, num_filters[1], n)
#         self.layer2 = self._make_layer(block, num_filters[2], n, stride=2)
#         self.layer3 = self._make_layer(block, num_filters[3], n, stride=2)
#         # self.avgpool = nn.AvgPool2d(8)
#         self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

#         self.fc = nn.Linear(num_filters[3] * block.expansion, num_classes)

#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#             elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
#                 nn.init.constant_(m.weight, 1)
#                 nn.init.constant_(m.bias, 0)

#     def _make_layer(self, block, planes, blocks, stride=1):
#         downsample = None
#         if stride != 1 or self.inplanes != planes * block.expansion:
#             downsample = nn.Sequential(
#                 nn.Conv2d(self.inplanes, planes * block.expansion,
#                           kernel_size=1, stride=stride, bias=False),
#                 nn.BatchNorm2d(planes * block.expansion),
#             )

#         layers = list([])
#         layers.append(block(self.inplanes, planes, stride, downsample, is_last=(blocks == 1)))
#         self.inplanes = planes * block.expansion
#         for i in range(1, blocks):
#             layers.append(block(self.inplanes, planes, is_last=(i == blocks-1)))

#         return nn.Sequential(*layers)

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.conv1)
#         feat_m.append(self.bn1)
#         feat_m.append(self.relu)
#         feat_m.append(self.layer1)
#         feat_m.append(self.layer2)
#         feat_m.append(self.layer3)
#         feat_m.append(self.fc)
#         return feat_m

#     def get_bn_before_relu(self):
#         if isinstance(self.layer1[0], Bottleneck):
#             bn1 = self.layer1[-1].bn3
#             bn2 = self.layer2[-1].bn3
#             bn3 = self.layer3[-1].bn3
#         elif isinstance(self.layer1[0], BasicBlock):
#             bn1 = self.layer1[-1].bn2
#             bn2 = self.layer2[-1].bn2
#             bn3 = self.layer3[-1].bn2
#         else:
#             raise NotImplementedError('ResNet unknown block error !!!')

#         return [bn1, bn2, bn3]

#     def forward(self, x, is_feat=False, preact=False):
#         x = self.conv1(x)
#         x = self.bn1(x)
#         x = self.relu(x)  # 32x32
#         f0 = x

#         x, f1_pre = self.layer1(x)  # 32x32
#         f1 = x
#         x, f2_pre = self.layer2(x)  # 16x16
#         f2 = x
#         x, f3_pre = self.layer3(x)  # 8x8
#         f3 = x

#         x = self.avgpool(x)
#         x = x.view(x.size(0), -1)
#         f4 = x
#         x = self.fc(x)

#         if is_feat:
#             if preact:
#                 return [f0, f1_pre, f2_pre, f3_pre, f4], x
#             else:
#                 return [f0, f1, f2, f3, f4], x
#         else:
#             return x


# def resnet8(**kwargs):
#     return ResNet(8, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet14(**kwargs):
#     return ResNet(14, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet20(**kwargs): # 41.6M
#     return ResNet(20, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet32(**kwargs): # 70.4M
#     return ResNet(32, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet44(**kwargs):
#     return ResNet(44, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet56(**kwargs):
#     return ResNet(56, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet110(**kwargs):
#     return ResNet(110, [16, 16, 32, 64], 'basicblock', **kwargs)


# def resnet8x4(**kwargs):
#     return ResNet(8, [32, 64, 128, 256], 'basicblock', **kwargs)


# def resnet8x4_double(**kwargs):
#     return ResNet(8, [64, 128, 256, 512], 'basicblock', **kwargs)


# def resnet32x4(**kwargs):
#     return ResNet(32, [32, 64, 128, 256], 'basicblock', **kwargs)


# def gap_loss(y_s, y_t, temp_stu, temp_tea):

#     loss = (torch.logsumexp(y_t/temp_tea, 1)-torch.logsumexp(y_s/temp_stu, 1)).mean(0)

#     return loss

# # if __name__ == '__main__':
# #     import torch
# #     from thop import clever_format, profile
# #     model = resnet32()
# #     input = torch.randn(1, 3, 32, 32)
# #     macs, params = profile(model, inputs=(input, ))
# #     macs, params = clever_format([macs, params], "%.3f")
# #     print(macs)
# #     print(params)

# # def resnet20(num_classes=100):  return ResNet(BasicBlock, [3, 3, 3], num_classes)
# # def resnet56(num_classes=100):  return ResNet(BasicBlock, [9, 9, 9], num_classes)
# # def resnet110(num_classes=100): return ResNet(BasicBlock, [18, 18, 18], num_classes)

# WRN Architecture

In [4]:


# """
# Original Author: Wei Yang
# """

# __all__ = ['wrn']


# class BasicBlockWRN(nn.Module):
#     def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
#         super(BasicBlockWRN, self).__init__()
#         self.bn1 = nn.BatchNorm2d(in_planes)
#         self.relu1 = nn.ReLU(inplace=True)
#         self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
#                                padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_planes)
#         self.relu2 = nn.ReLU(inplace=True)
#         self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1,
#                                padding=1, bias=False)
#         self.droprate = dropRate
#         self.equalInOut = (in_planes == out_planes)
#         self.convShortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride,
#                                padding=0, bias=False) or None

#     def forward(self, x):
#         if not self.equalInOut:
#             x = self.relu1(self.bn1(x))
#         else:
#             out = self.relu1(self.bn1(x))
#         out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
#         if self.droprate > 0:
#             out = F.dropout(out, p=self.droprate, training=self.training)
#         out = self.conv2(out)
#         return torch.add(x if self.equalInOut else self.convShortcut(x), out)


# class NetworkBlock(nn.Module):
#     def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
#         super(NetworkBlock, self).__init__()
#         self.layer = self._make_layer(block, in_planes, out_planes, nb_layers, stride, dropRate)

#     def _make_layer(self, block, in_planes, out_planes, nb_layers, stride, dropRate):
#         layers = []
#         for i in range(nb_layers):
#             layers.append(block(i == 0 and in_planes or out_planes, out_planes, i == 0 and stride or 1, dropRate))
#         return nn.Sequential(*layers)

#     def forward(self, x):
#         return self.layer(x)


# class WideResNet(nn.Module):
#     def __init__(self, depth, num_classes=100, widen_factor=1, dropRate=0.0):
#         super(WideResNet, self).__init__()
#         nChannels = [16, 16*widen_factor, 32*widen_factor, 64*widen_factor]
#         assert (depth - 4) % 6 == 0, 'depth should be 6n+4'
#         n = (depth - 4) // 6
#         block = BasicBlockWRN
#         # 1st conv before any network block
#         self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1,
#                                padding=1, bias=False)
#         # 1st block
#         self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
#         # 2nd block
#         self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
#         # 3rd block
#         self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
#         # global average pooling and classifier
#         self.bn1 = nn.BatchNorm2d(nChannels[3])
#         self.relu = nn.ReLU(inplace=True)
#         self.fc = nn.Linear(nChannels[3], num_classes)
#         self.nChannels = nChannels[3]

#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
#                 m.weight.data.normal_(0, math.sqrt(2. / n))
#             elif isinstance(m, nn.BatchNorm2d):
#                 m.weight.data.fill_(1)
#                 m.bias.data.zero_()
#             elif isinstance(m, nn.Linear):
#                 m.bias.data.zero_()

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.conv1)
#         feat_m.append(self.block1)
#         feat_m.append(self.block2)
#         feat_m.append(self.block3)
#         feat_m.append(self.fc)
#         return feat_m

#     def get_bn_before_relu(self):
#         bn1 = self.block2.layer[0].bn1
#         bn2 = self.block3.layer[0].bn1
#         bn3 = self.bn1

#         return [bn1, bn2, bn3]

#     def forward(self, x, is_feat=False, preact=False):
#         out = self.conv1(x)
#         f0 = out
#         out = self.block1(out)
#         f1 = out
#         out = self.block2(out)
#         f2 = out
#         out = self.block3(out)
#         f3 = out
#         out = self.relu(self.bn1(out))
#         # out = F.avg_pool2d(out, 8)
#         out = F.adaptive_avg_pool2d(out, (1, 1))
#         out = out.view(-1, self.nChannels)
#         f4 = out
#         out = self.fc(out)
#         if is_feat:
#             if preact:
#                 f1 = self.block2.layer[0].bn1(f1)
#                 f2 = self.block3.layer[0].bn1(f2)
#                 f3 = self.bn1(f3)
#             return [f0, f1, f2, f3, f4], out
#         else:
#             return out


# def wrn(**kwargs):
#     """
#     Constructs a Wide Residual Networks.
#     """
#     model = WideResNet(**kwargs)
#     return model


# def wrn_40_2(**kwargs):
#     model = WideResNet(depth=40, widen_factor=2, **kwargs)
#     return model


# def wrn_40_1(**kwargs):
#     model = WideResNet(depth=40, widen_factor=1, **kwargs)
#     return model


# def wrn_16_2(**kwargs):
#     model = WideResNet(depth=16, widen_factor=2, **kwargs)
#     return model


# def wrn_16_1(**kwargs):
#     model = WideResNet(depth=16, widen_factor=1, **kwargs)
#     return model


# class CosineIncrease(object):
#     def __init__(self,
#                 max_value,
#                 min_value,
#                 num_loops):
#         self._max_value = max_value # 1
#         self._min_value = min_value # 0
#         self._num_loops = num_loops # 10

#     def get_value(self, i):
#         if i < 0:
#             i = 0
#         if i >= self._num_loops:
#             i = self._num_loops
#         value = (math.cos((1 + i / self._num_loops) * math.pi) + 1.0) * 0.5
#         value = value * (self._max_value - self._min_value) + self._min_value
#         return value

# # if __name__ == '__main__':
# #     # import torch
# #     # from thop import profile
# #     # from thop import clever_format
# #     # model = wrn_40_1()
# #     # input = torch.randn(1, 3, 32, 32)
# #     # macs, params = profile(model, inputs=(input, ))
# #     # macs, params = clever_format([macs, params], "%.3f")
# #     # print(macs)
# #     # print(params)

# #     cosine_decay = CosineDecay(1, 0, 10)
# #     for i in range(20):
# #         current = cosine_decay.get_value(i)
# #         print(current)



# MobileNetV2 Architecture

In [5]:
# """
# MobileNetV2 implementation used in
# <Knowledge Distillation via Route Constrained Optimization>
# """

# __all__ = ['mobilenetv2_T_w', 'mobile_half']

# BN = None


# def conv_bn(inp, oup, stride):
#     return nn.Sequential(
#         nn.Conv2d(inp, oup, 3, stride, 1, bias=False),
#         nn.BatchNorm2d(oup),
#         nn.ReLU(inplace=True)
#     )


# def conv_1x1_bn(inp, oup):
#     return nn.Sequential(
#         nn.Conv2d(inp, oup, 1, 1, 0, bias=False),
#         nn.BatchNorm2d(oup),
#         nn.ReLU(inplace=True)
#     )


# class InvertedResidual(nn.Module):
#     def __init__(self, inp, oup, stride, expand_ratio):
#         super(InvertedResidual, self).__init__()
#         self.blockname = None

#         self.stride = stride
#         assert stride in [1, 2]

#         self.use_res_connect = self.stride == 1 and inp == oup

#         self.conv = nn.Sequential(
#             # pw
#             nn.Conv2d(inp, inp * expand_ratio, 1, 1, 0, bias=False),
#             nn.BatchNorm2d(inp * expand_ratio),
#             nn.ReLU(inplace=True),
#             # dw
#             nn.Conv2d(inp * expand_ratio, inp * expand_ratio, 3, stride, 1, groups=inp * expand_ratio, bias=False),
#             nn.BatchNorm2d(inp * expand_ratio),
#             nn.ReLU(inplace=True),
#             # pw-linear
#             nn.Conv2d(inp * expand_ratio, oup, 1, 1, 0, bias=False),
#             nn.BatchNorm2d(oup),
#         )
#         self.names = ['0', '1', '2', '3', '4', '5', '6', '7']

#     def forward(self, x):
#         t = x
#         if self.use_res_connect:
#             return t + self.conv(x)
#         else:
#             return self.conv(x)


# class MobileNetV2(nn.Module):
#     """mobilenetV2"""
#     def __init__(self, T,
#                  feature_dim,
#                  input_size=32,
#                  width_mult=1.,
#                  remove_avg=False):
#         super(MobileNetV2, self).__init__()
#         self.remove_avg = remove_avg

#         # setting of inverted residual blocks
#         self.interverted_residual_setting = [
#             # t, c, n, s
#             [1, 16, 1, 1],
#             [T, 24, 2, 1],
#             [T, 32, 3, 2],
#             [T, 64, 4, 2],
#             [T, 96, 3, 1],
#             [T, 160, 3, 2],
#             [T, 320, 1, 1],
#         ]

#         # building first layer
#         assert input_size % 32 == 0
#         input_channel = int(32 * width_mult)
#         self.conv1 = conv_bn(3, input_channel, 2)

#         # building inverted residual blocks
#         self.blocks = nn.ModuleList([])
#         for t, c, n, s in self.interverted_residual_setting:
#             output_channel = int(c * width_mult)
#             layers = []
#             strides = [s] + [1] * (n - 1)
#             for stride in strides:
#                 layers.append(
#                     InvertedResidual(input_channel, output_channel, stride, t)
#                 )
#                 input_channel = output_channel
#             self.blocks.append(nn.Sequential(*layers))

#         self.last_channel = int(1280 * width_mult) if width_mult > 1.0 else 1280
#         self.conv2 = conv_1x1_bn(input_channel, self.last_channel)

#         # building classifier
#         self.classifier = nn.Sequential(
#             # nn.Dropout(0.5),
#             nn.Linear(self.last_channel, feature_dim),
#         )

#         H = input_size // (32//2)
#         self.avgpool = nn.AvgPool2d(H, ceil_mode=True)

#         self._initialize_weights()
#         print(T, width_mult)

#     def get_bn_before_relu(self):
#         bn1 = self.blocks[1][-1].conv[-1]
#         bn2 = self.blocks[2][-1].conv[-1]
#         bn3 = self.blocks[4][-1].conv[-1]
#         bn4 = self.blocks[6][-1].conv[-1]
#         return [bn1, bn2, bn3, bn4]

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.conv1)
#         feat_m.append(self.blocks)
#         # feat_m.append(self.conv2)
#         # feat_m.append(self.classifier)
#         return feat_m

#     def forward(self, x, is_feat=False, preact=False):

#         out = self.conv1(x)
#         f0 = out

#         out = self.blocks[0](out)
#         out = self.blocks[1](out)
#         f1 = out
#         out = self.blocks[2](out)
#         f2 = out
#         out = self.blocks[3](out)
#         out = self.blocks[4](out)
#         f3 = out
#         out = self.blocks[5](out)
#         out = self.blocks[6](out)
#         f4 = out

#         out = self.conv2(out)

#         if not self.remove_avg:
#             out = self.avgpool(out)
#         out = out.view(out.size(0), -1)
#         f5 = out
#         # print(f5.shape)
#         out = self.classifier(out)
#         # print(out.shape)

#         if is_feat:
#             return [f0, f1, f2, f3, f4, f5], out
#         else:
#             return out

#     def _initialize_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
#                 m.weight.data.normal_(0, math.sqrt(2. / n))
#                 if m.bias is not None:
#                     m.bias.data.zero_()
#             elif isinstance(m, nn.BatchNorm2d):
#                 m.weight.data.fill_(1)
#                 m.bias.data.zero_()
#             elif isinstance(m, nn.Linear):
#                 n = m.weight.size(1)
#                 m.weight.data.normal_(0, 0.01)
#                 m.bias.data.zero_()


# def mobilenetv2_T_w(T, W, feature_dim=100):
#     model = MobileNetV2(T=T, feature_dim=feature_dim, width_mult=W)
#     return model


# def mobile_half(num_classes):
#     return mobilenetv2_T_w(6, 0.5, num_classes)


# # if __name__ == '__main__':
# #     x = torch.randn(2, 3, 32, 32)

# #     net = mobile_half(100)

# #     feats, logit = net(x, is_feat=True, preact=True)
# #     for f in feats:
# #         print(f.shape, f.min().item())
# #     print(logit.shape)

# #     for m in net.get_bn_before_relu():
# #         if isinstance(m, nn.BatchNorm2d):
# #             print('pass')
# #         else:
# #             print('warning')


# ShuffleNet V1 Architecture

In [6]:
# '''ShuffleNet in PyTorch.
# See the paper "ShuffleNet: An Extremely Efficient Convolutional Neural Network for Mobile Devices" for more details.
# '''
# import torch
# import torch.nn as nn
# import torch.nn.functional as F


# class ShuffleBlock(nn.Module):
#     def __init__(self, groups):
#         super(ShuffleBlock, self).__init__()
#         self.groups = groups

#     def forward(self, x):
#         '''Channel shuffle: [N,C,H,W] -> [N,g,C/g,H,W] -> [N,C/g,g,H,w] -> [N,C,H,W]'''
#         N,C,H,W = x.size()
#         g = self.groups
#         return x.view(N,g,C//g,H,W).permute(0,2,1,3,4).reshape(N,C,H,W)


# class Bottleneck(nn.Module):
#     def __init__(self, in_planes, out_planes, stride, groups, is_last=False):
#         super(Bottleneck, self).__init__()
#         self.is_last = is_last
#         self.stride = stride

#         mid_planes = int(out_planes/4)
#         g = 1 if in_planes == 24 else groups
#         self.conv1 = nn.Conv2d(in_planes, mid_planes, kernel_size=1, groups=g, bias=False)
#         self.bn1 = nn.BatchNorm2d(mid_planes)
#         self.shuffle1 = ShuffleBlock(groups=g)
#         self.conv2 = nn.Conv2d(mid_planes, mid_planes, kernel_size=3, stride=stride, padding=1, groups=mid_planes, bias=False)
#         self.bn2 = nn.BatchNorm2d(mid_planes)
#         self.conv3 = nn.Conv2d(mid_planes, out_planes, kernel_size=1, groups=groups, bias=False)
#         self.bn3 = nn.BatchNorm2d(out_planes)

#         self.shortcut = nn.Sequential()
#         if stride == 2:
#             self.shortcut = nn.Sequential(nn.AvgPool2d(3, stride=2, padding=1))

#     def forward(self, x):
#         out = F.relu(self.bn1(self.conv1(x)))
#         out = self.shuffle1(out)
#         out = F.relu(self.bn2(self.conv2(out)))
#         out = self.bn3(self.conv3(out))
#         res = self.shortcut(x)
#         preact = torch.cat([out, res], 1) if self.stride == 2 else out+res
#         out = F.relu(preact)
#         # out = F.relu(torch.cat([out, res], 1)) if self.stride == 2 else F.relu(out+res)
#         if self.is_last:
#             return out, preact
#         else:
#             return out


# class ShuffleNet(nn.Module):
#     def __init__(self, cfg, num_classes=100):
#         super(ShuffleNet, self).__init__()
#         out_planes = cfg['out_planes']
#         num_blocks = cfg['num_blocks']
#         groups = cfg['groups']

#         self.conv1 = nn.Conv2d(3, 24, kernel_size=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(24)
#         self.in_planes = 24
#         self.layer1 = self._make_layer(out_planes[0], num_blocks[0], groups)
#         self.layer2 = self._make_layer(out_planes[1], num_blocks[1], groups)
#         self.layer3 = self._make_layer(out_planes[2], num_blocks[2], groups)
#         self.linear = nn.Linear(out_planes[2], num_classes)

#     def _make_layer(self, out_planes, num_blocks, groups):
#         layers = []
#         for i in range(num_blocks):
#             stride = 2 if i == 0 else 1
#             cat_planes = self.in_planes if i == 0 else 0
#             layers.append(Bottleneck(self.in_planes, out_planes-cat_planes,
#                                      stride=stride,
#                                      groups=groups,
#                                      is_last=(i == num_blocks - 1)))
#             self.in_planes = out_planes
#         return nn.Sequential(*layers)

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.conv1)
#         feat_m.append(self.bn1)
#         feat_m.append(self.layer1)
#         feat_m.append(self.layer2)
#         feat_m.append(self.layer3)
#         return feat_m

#     def get_bn_before_relu(self):
#         raise NotImplementedError('ShuffleNet currently is not supported for "Overhaul" teacher')

#     def forward(self, x, is_feat=False, preact=False):
#         out = F.relu(self.bn1(self.conv1(x)))
#         f0 = out
#         out, f1_pre = self.layer1(out)
#         f1 = out
#         out, f2_pre = self.layer2(out)
#         f2 = out
#         out, f3_pre = self.layer3(out)
#         f3 = out
#         # out = F.avg_pool2d(out, 4)
#         """
#         Making it adaptive
#         """
#         out = F.adaptive_avg_pool2d(out, 1)
#         out = out.view(out.size(0), -1)
#         f4 = out
#         out = self.linear(out)

#         if is_feat:
#             if preact:
#                 return [f0, f1_pre, f2_pre, f3_pre, f4], out
#             else:
#                 return [f0, f1, f2, f3, f4], out
#         else:
#             return out


# def ShuffleV1(**kwargs):
#     cfg = {
#         'out_planes': [240, 480, 960],
#         'num_blocks': [4, 8, 4],
#         'groups': 3
#     }
#     return ShuffleNet(cfg, **kwargs)


# # if __name__ == '__main__':

# #     x = torch.randn(2, 3, 32, 32)
# #     net = ShuffleV1(num_classes=100)
# #     import time
# #     a = time.time()
# #     feats, logit = net(x, is_feat=True, preact=True)
# #     b = time.time()
# #     print(b - a)
# #     for f in feats:
# #         print(f.shape, f.min().item())
# #     print(logit.shape)

# Mapping VGG

# VGG Architecture

In [7]:
'''VGG for CIFAR10. FC layers are removed.
(c) YANG, Wei
'''
import torch.nn as nn
import torch.nn.functional as F
import math


__all__ = [
    'VGG', 'vgg11', 'vgg11_bn', 'vgg13', 'vgg13_bn', 'vgg16', 'vgg16_bn',
    'vgg19_bn', 'vgg19',
]


model_urls = {
    'vgg11': 'https://download.pytorch.org/models/vgg11-bbd30ac9.pth',
    'vgg13': 'https://download.pytorch.org/models/vgg13-c768596a.pth',
    'vgg16': 'https://download.pytorch.org/models/vgg16-397923af.pth',
    'vgg19': 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
}


class VGG(nn.Module):

    def __init__(self, cfg, batch_norm=False, num_classes=100):
        super(VGG, self).__init__()
        self.block0 = self._make_layers(cfg[0], batch_norm, 3)
        self.block1 = self._make_layers(cfg[1], batch_norm, cfg[0][-1])
        self.block2 = self._make_layers(cfg[2], batch_norm, cfg[1][-1])
        self.block3 = self._make_layers(cfg[3], batch_norm, cfg[2][-1])
        self.block4 = self._make_layers(cfg[4], batch_norm, cfg[3][-1])

        self.pool0 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool4 = nn.AdaptiveAvgPool2d((1, 1))
        # self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.classifier = nn.Linear(512, num_classes)
        self._initialize_weights()

    def get_feat_modules(self):
        feat_m = nn.ModuleList([])
        feat_m.append(self.block0)
        feat_m.append(self.pool0)
        feat_m.append(self.block1)
        feat_m.append(self.pool1)
        feat_m.append(self.block2)
        feat_m.append(self.pool2)
        feat_m.append(self.block3)
        feat_m.append(self.pool3)
        feat_m.append(self.block4)
        feat_m.append(self.pool4)
        return feat_m

    def get_bn_before_relu(self):
        bn1 = self.block1[-1]
        bn2 = self.block2[-1]
        bn3 = self.block3[-1]
        bn4 = self.block4[-1]
        return [bn1, bn2, bn3, bn4]

    def forward(self, x, is_feat=False, preact=False):
        h = x.shape[2]
        x = F.relu(self.block0(x))
        f0 = x
        x = self.pool0(x)
        x = self.block1(x)
        f1_pre = x
        x = F.relu(x)
        f1 = x
        x = self.pool1(x)
        x = self.block2(x)
        f2_pre = x
        x = F.relu(x)
        f2 = x
        x = self.pool2(x)
        x = self.block3(x)
        f3_pre = x
        x = F.relu(x)
        f3 = x
        if h == 64:
            x = self.pool3(x)
        x = self.block4(x)
        f4_pre = x
        x = F.relu(x)
        f4 = x
        x = self.pool4(x)
        x = x.view(x.size(0), -1)
        f5 = x
        x = self.classifier(x)

        if is_feat:
            if preact:
                return [f0, f1_pre, f2_pre, f3_pre, f4_pre, f5], x
            else:
                return [f0, f1, f2, f3, f4, f5], x
        else:
            return x

    @staticmethod
    def _make_layers(cfg, batch_norm=False, in_channels=3):
        layers = []
        for v in cfg:
            if v == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                if batch_norm:
                    layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                else:
                    layers += [conv2d, nn.ReLU(inplace=True)]
                in_channels = v
        layers = layers[:-1]
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                n = m.weight.size(1)
                m.weight.data.normal_(0, 0.01)
                m.bias.data.zero_()


cfg = {
    'A': [[64], [128], [256, 256], [512, 512], [512, 512]],
    'B': [[64, 64], [128, 128], [256, 256], [512, 512], [512, 512]],
    'D': [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]],
    'E': [[64, 64], [128, 128], [256, 256, 256, 256], [512, 512, 512, 512], [512, 512, 512, 512]],
    'S': [[64], [128], [256], [512], [512]],
}


def vgg8(**kwargs):
    """VGG 8-layer model (configuration "S")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['S'], **kwargs)
    return model


def vgg8_bn(**kwargs):
    """VGG 8-layer model (configuration "S")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['S'], batch_norm=True, **kwargs)
    return model


def vgg11(**kwargs):
    """VGG 11-layer model (configuration "A")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['A'], **kwargs)
    return model


def vgg11_bn(**kwargs):
    """VGG 11-layer model (configuration "A") with batch normalization"""
    model = VGG(cfg['A'], batch_norm=True, **kwargs)
    return model


def vgg13(**kwargs):
    """VGG 13-layer model (configuration "B")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['B'], **kwargs)
    return model


def vgg13_bn(**kwargs):
    """VGG 13-layer model (configuration "B") with batch normalization"""
    model = VGG(cfg['B'], batch_norm=True, **kwargs)
    return model


def vgg16(**kwargs):
    """VGG 16-layer model (configuration "D")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['D'], **kwargs)
    return model


def vgg16_bn(**kwargs):
    """VGG 16-layer model (configuration "D") with batch normalization"""
    model = VGG(cfg['D'], batch_norm=True, **kwargs)
    return model


def vgg19(**kwargs):
    """VGG 19-layer model (configuration "E")
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = VGG(cfg['E'], **kwargs)
    return model


def vgg19_bn(**kwargs):
    """VGG 19-layer model (configuration 'E') with batch normalization"""
    model = VGG(cfg['E'], batch_norm=True, **kwargs)
    return model


# if __name__ == '__main__':
#     import torch

#     x = torch.randn(2, 3, 32, 32)
#     net = vgg19_bn(num_classes=100)
#     feats, logit = net(x, is_feat=True, preact=True)

#     for f in feats:
#         print(f.shape, f.min().item())
#     print(logit.shape)

#     for m in net.get_bn_before_relu():
#         if isinstance(m, nn.BatchNorm2d):
#             print('pass')
#         else:
#             print('warning')

In [8]:
# --- mdistiller ImageNet ResNet (DKD paper source, adaptive avgpool patch) ---
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.hub import load_state_dict_from_url

__all__ = ["ResNet", "resnet18", "resnet34"]

_IMAGENET_RESNET_URLS = {
    "resnet18": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    "resnet34": "https://download.pytorch.org/models/resnet34-333f7ec4.pth",
}


def _conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = _conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = _conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        x = F.relu(x)
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        x = F.relu(x)
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2.0 / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        stem = x
        x = self.relu(x)
        x = self.maxpool(x)

        feat1 = self.layer1(x)
        feat2 = self.layer2(feat1)
        feat3 = self.layer3(feat2)
        feat4 = self.layer4(feat3)

        pooled = self.avgpool(F.relu(feat4))
        pooled = pooled.view(pooled.size(0), -1)
        out = self.fc(pooled)

        feats = {
            "pooled_feat": pooled,
            "feats": [F.relu(stem), F.relu(feat1), F.relu(feat2), F.relu(feat3), F.relu(feat4)],
            "preact_feats": [stem, feat1, feat2, feat3, feat4],
        }
        return out, feats


def _maybe_load_pretrained(model, model_name, checkpoint_dir):
    ckpt_dir = checkpoint_dir if checkpoint_dir else os.path.join(".", ".cache", "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    state = load_state_dict_from_url(
        _IMAGENET_RESNET_URLS[model_name],
        model_dir=ckpt_dir,
        progress=True,
        map_location="cpu",
    )
    model.load_state_dict(state)
    return model


def resnet18(pretrained=False, checkpoint_dir="./.cache/checkpoints", **kwargs):
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    if pretrained:
        model = _maybe_load_pretrained(model, "resnet18", checkpoint_dir)
    return model


def resnet34(pretrained=False, checkpoint_dir="./.cache/checkpoints", **kwargs):
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    if pretrained:
        model = _maybe_load_pretrained(model, "resnet34", checkpoint_dir)
    return model


In [9]:
# '''VGG for CIFAR10. FC layers are removed.
# (c) YANG, Wei
# Modified for Inter-Resolution Knowledge Distillation with adaptive kernels
# '''
# import torch.nn as nn
# import torch.nn.functional as F
# import math
# import torch


# __all__ = [
#     'VGG', 'vgg11', 'vgg11_bn', 'vgg13', 'vgg13_bn', 'vgg16', 'vgg16_bn',
#     'vgg19_bn', 'vgg19', 'vgg8', 'vgg8_bn'
# ]


# model_urls = {
#     'vgg11': 'https://download.pytorch.org/models/vgg11-bbd30ac9.pth',
#     'vgg13': 'https://download.pytorch.org/models/vgg13-c768596a.pth',
#     'vgg16': 'https://download.pytorch.org/models/vgg16-397923af.pth',
#     'vgg19': 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
# }


# class VGG(nn.Module):

#     def __init__(self, cfg, batch_norm=False, num_classes=100, target_resolution=32, base_resolution=32):
#         super(VGG, self).__init__()
        
#         # Calculate adaptive kernel size for resolution curriculum
#         self.target_resolution = target_resolution
#         self.base_resolution = base_resolution
#         self.scale_factor = target_resolution / base_resolution
        
#         # Calculate adaptive kernel size (minimum 1x1, maximum 3x3)
#         self.adaptive_kernel = max(1, min(3, int(3 * self.scale_factor)))
#         self.adaptive_padding = self.adaptive_kernel // 2
        
#         print(f"VGG Resolution: {target_resolution}x{target_resolution}, "
#               f"Scale: {self.scale_factor:.2f}, "
#               f"Kernel: {self.adaptive_kernel}x{self.adaptive_kernel}")
        
#         self.block0 = self._make_layers(cfg[0], batch_norm, 3)
#         self.block1 = self._make_layers(cfg[1], batch_norm, cfg[0][-1])
#         self.block2 = self._make_layers(cfg[2], batch_norm, cfg[1][-1])
#         self.block3 = self._make_layers(cfg[3], batch_norm, cfg[2][-1])
#         self.block4 = self._make_layers(cfg[4], batch_norm, cfg[3][-1])

#         self.pool0 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.pool4 = nn.AdaptiveAvgPool2d((1, 1))

#         self.classifier = nn.Linear(512, num_classes)
#         self._initialize_weights()

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.block0)
#         feat_m.append(self.pool0)
#         feat_m.append(self.block1)
#         feat_m.append(self.pool1)
#         feat_m.append(self.block2)
#         feat_m.append(self.pool2)
#         feat_m.append(self.block3)
#         feat_m.append(self.pool3)
#         feat_m.append(self.block4)
#         feat_m.append(self.pool4)
#         return feat_m

#     def get_bn_before_relu(self):
#         bn1 = self.block1[-1]
#         bn2 = self.block2[-1]
#         bn3 = self.block3[-1]
#         bn4 = self.block4[-1]
#         return [bn1, bn2, bn3, bn4]

#     def forward(self, x, is_feat=False, preact=False):
#         h = x.shape[2]
#         x = F.relu(self.block0(x))
#         f0 = x
#         x = self.pool0(x)
#         x = self.block1(x)
#         f1_pre = x
#         x = F.relu(x)
#         f1 = x
#         x = self.pool1(x)
#         x = self.block2(x)
#         f2_pre = x
#         x = F.relu(x)
#         f2 = x
#         x = self.pool2(x)
#         x = self.block3(x)
#         f3_pre = x
#         x = F.relu(x)
#         f3 = x
#         if h == 64:
#             x = self.pool3(x)
#         x = self.block4(x)
#         f4_pre = x
#         x = F.relu(x)
#         f4 = x
#         x = self.pool4(x)
#         x = x.view(x.size(0), -1)
#         f5 = x
#         x = self.classifier(x)

#         if is_feat:
#             if preact:
#                 return [f0, f1_pre, f2_pre, f3_pre, f4_pre, f5], x
#             else:
#                 return [f0, f1, f2, f3, f4, f5], x
#         else:
#             return x

#     def _make_layers(self, cfg, batch_norm=False, in_channels=3):
#         layers = []
#         for v in cfg:
#             if v == 'M':
#                 layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
#             else:
#                 # Use adaptive kernel size instead of fixed 3x3
#                 conv2d = nn.Conv2d(in_channels, v, 
#                                  kernel_size=self.adaptive_kernel, 
#                                  padding=self.adaptive_padding)
#                 if batch_norm:
#                     layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
#                 else:
#                     layers += [conv2d, nn.ReLU(inplace=True)]
#                 in_channels = v
#         layers = layers[:-1]
#         return nn.Sequential(*layers)

#     def _initialize_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
#                 m.weight.data.normal_(0, math.sqrt(2. / n))
#                 if m.bias is not None:
#                     m.bias.data.zero_()
#             elif isinstance(m, nn.BatchNorm2d):
#                 m.weight.data.fill_(1)
#                 m.bias.data.zero_()
#             elif isinstance(m, nn.Linear):
#                 n = m.weight.size(1)
#                 m.weight.data.normal_(0, 0.01)
#                 m.bias.data.zero_()


# def transfer_all_weights(source_model, target_model, method='interpolate'):
#     """
#     Transfer ALL compatible weights from source to target model
#     """
#     source_params = dict(source_model.named_parameters())
#     target_params = dict(target_model.named_parameters())
#     source_buffers = dict(source_model.named_buffers())
#     target_buffers = dict(target_model.named_buffers())
    
#     conv_transferred = 0
#     direct_transferred = 0
#     buffer_transferred = 0
    
#     # Transfer parameters
#     for name, target_param in target_params.items():
#         if name in source_params:
#             source_param = source_params[name]
            
#             # Check if this is a conv layer weight (4D tensor with .weight in name)
#             if target_param.dim() == 4 and '.weight' in name:
#                 target_shape = target_param.shape
#                 source_shape = source_param.shape
                
#                 # Check if channel dimensions match (in_channels, out_channels)
#                 if target_shape[:2] == source_shape[:2]:
#                     print(f"Processing conv layer: {name}")
#                     print(f"  Source shape: {source_shape}, Target shape: {target_shape}")
                    
#                     if method == 'interpolate':
#                         with torch.no_grad():
#                             # Reshape for interpolation: (out_ch * in_ch, 1, H, W)
#                             source_reshaped = source_param.view(-1, 1, source_shape[2], source_shape[3])
#                             target_reshaped = F.interpolate(
#                                 source_reshaped, 
#                                 size=(target_shape[2], target_shape[3]), 
#                                 mode='bilinear', 
#                                 align_corners=False
#                             )
#                             target_param.data = target_reshaped.view(target_shape)
#                             conv_transferred += 1
#                             print(f"  ✓ Interpolated {source_shape[2]}x{source_shape[3]} → {target_shape[2]}x{target_shape[3]}")
                    
#                     elif method == 'center_pad':
#                         with torch.no_grad():
#                             target_param.data.zero_()
#                             if source_shape[2] <= target_shape[2] and source_shape[3] <= target_shape[3]:
#                                 h_offset = (target_shape[2] - source_shape[2]) // 2
#                                 w_offset = (target_shape[3] - source_shape[3]) // 2
#                                 target_param.data[:, :, 
#                                                 h_offset:h_offset+source_shape[2],
#                                                 w_offset:w_offset+source_shape[3]] = source_param.data
#                                 conv_transferred += 1
#                                 print(f"  ✓ Center-padded {source_shape[2]}x{source_shape[3]} → {target_shape[2]}x{target_shape[3]}")
#                 else:
#                     print(f"  ✗ Skipping {name}: channel mismatch {source_shape[:2]} vs {target_shape[:2]}")
            
#             # All other parameters (bias, BatchNorm, Linear) - direct copy if shapes match
#             elif target_param.shape == source_param.shape:
#                 with torch.no_grad():
#                     target_param.data.copy_(source_param.data)
#                     direct_transferred += 1
#                     print(f"Direct transfer: {name} {target_param.shape}")
#             else:
#                 print(f"Shape mismatch for {name}: {source_param.shape} vs {target_param.shape}")
    
#     # Transfer buffers (BatchNorm running stats, etc.)
#     for name, target_buffer in target_buffers.items():
#         if name in source_buffers:
#             source_buffer = source_buffers[name]
#             if target_buffer.shape == source_buffer.shape:
#                 with torch.no_grad():
#                     target_buffer.data.copy_(source_buffer.data)
#                     buffer_transferred += 1
#                     print(f"Buffer transfer: {name} {target_buffer.shape}")
    
#     print(f"\n📊 SUMMARY: {conv_transferred} conv layers, {direct_transferred} other params, {buffer_transferred} buffers")
#     return conv_transferred, direct_transferred, buffer_transferred

# # Keep original cfg unchanged
# cfg = {
#     'A': [[64], [128], [256, 256], [512, 512], [512, 512]],
#     'B': [[64, 64], [128, 128], [256, 256], [512, 512], [512, 512]],
#     'D': [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]],
#     'E': [[64, 64], [128, 128], [256, 256, 256, 256], [512, 512, 512, 512], [512, 512, 512, 512]],
#     'S': [[64], [128], [256], [512], [512]],
# }


# def vgg8(**kwargs):
#     """VGG 8-layer model (configuration "S")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['S'], **kwargs)
#     return model


# def vgg8_bn(**kwargs):
#     """VGG 8-layer model (configuration "S")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['S'], batch_norm=True, **kwargs)
#     return model


# def vgg11(**kwargs):
#     """VGG 11-layer model (configuration "A")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['A'], **kwargs)
#     return model


# def vgg11_bn(**kwargs):
#     """VGG 11-layer model (configuration "A") with batch normalization"""
#     model = VGG(cfg['A'], batch_norm=True, **kwargs)
#     return model


# def vgg13(**kwargs):
#     """VGG 13-layer model (configuration "B")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['B'], **kwargs)
#     return model


# def vgg13_bn(**kwargs):
#     """VGG 13-layer model (configuration "B") with batch normalization"""
#     model = VGG(cfg['B'], batch_norm=True, **kwargs)
#     return model


# def vgg16(**kwargs):
#     """VGG 16-layer model (configuration "D")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['D'], **kwargs)
#     return model


# def vgg16_bn(**kwargs):
#     """VGG 16-layer model (configuration "D") with batch normalization"""
#     model = VGG(cfg['D'], batch_norm=True, **kwargs)
#     return model


# def vgg19(**kwargs):
#     """VGG 19-layer model (configuration "E")
#     Args:
#         pretrained (bool): If True, returns a model pre-trained on ImageNet
#     """
#     model = VGG(cfg['E'], **kwargs)
#     return model


# def vgg19_bn(**kwargs):
#     """VGG 19-layer model (configuration 'E') with batch normalization"""
#     model = VGG(cfg['E'], batch_norm=True, **kwargs)
#     return model


# # Example usage for IRKD curriculum
# if __name__ == '__main__':
#     # Create students for different resolutions using different VGG variants
#     resolutions = [16, 20, 24, 28, 32]
    
#     print("Creating VGG8 students for resolution curriculum:")
#     students_vgg8 = {}
#     for res in resolutions:
#         students_vgg8[res] = vgg8_bn(target_resolution=res, num_classes=100)
    
#     # print("\nCreating VGG16 students for resolution curriculum:")
#     # students_vgg16 = {}
#     # for res in resolutions:
#     #     students_vgg16[res] = vgg16_bn(target_resolution=res, num_classes=100)
    
#     # Test weight transfer between different resolutions
#     print("\n" + "="*60)
#     print("Testing weight transfer from 16x16 to 20x20 VGG8:")
#     print(students_vgg8[20])
#     transfer_all_weights(students_vgg8[20], students_vgg8[24], method='interpolate')
    
#     # print("\nTesting weight transfer from 16x16 to 20x20 VGG16:")
#     # transfer_all_weights(students_vgg16[16], students_vgg16[20], method='interpolate')
    
#     # Test forward pass
#     # x16 = torch.randn(4, 3, 16, 16)
#     # x32 = torch.randn(4, 3, 32, 32)
    
#     # output16 = students_vgg8[16](x16)
#     # output32 = students_vgg8[32](x32)
    
#     # print(f"\nVGG8 output shape for 16x16 input: {output16.shape}")
#     # print(f"VGG8 output shape for 32x32 input: {output32.shape}")
    
#     # # Show parameter counts for different resolutions
#     # print(f"\nVGG8 16x16 parameters: {sum(p.numel() for p in students_vgg8[16].parameters()):,}")
#     # print(f"VGG8 32x32 parameters: {sum(p.numel() for p in students_vgg8[32].parameters()):,}")

# Evaluating Results for ResNet and VGG
- ResNet32x4
- ResNet56
- ResNet110

In [10]:
# import matplotlib.pyplot as plt
# # ---------------------------
# # Device
# # ---------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ---------------------------
# # CIFAR-100 test dataset
# # ---------------------------
# mean = (0.5071, 0.4867, 0.4408)
# std  = (0.2675, 0.2565, 0.2761)

# test_tf = transforms.Compose([
#     transforms.RandomCrop(32),
#     transforms.ToTensor(),
#     transforms.Normalize(mean, std)
# ])

# test_ds = datasets.CIFAR100("./data", train=False, transform=test_tf, download=True)
# test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# # ---------------------------
# # Model constructors
# # ---------------------------
# model_dict = {
#     "VGG13": vgg13_bn(),
#     "ResNet32x4": resnet32x4(), # this is the resnet-32x4
#     "ResNet56": resnet56(),
#     "ResNet110": resnet110()
# }

# # Path mapping for pretrained weights (adjust paths if needed)
# weights_dict = {
#     "VGG13": "/kaggle/input/vgg_13/pytorch/default/1/ckpt_epoch_240.pth",
#     "ResNet32x4": "/kaggle/input/resnet32x4/pytorch/default/1/ckpt_epoch_240.pth",
#     "ResNet56": "/kaggle/input/resnet_56/pytorch/default/1/ckpt_epoch_240.pth",
#     "ResNet110": "/kaggle/input/resnet110/pytorch/default/1/ckpt_epoch_240.pth"
# }

# # ---------------------------
# # Evaluation loop
# # ---------------------------
# for name, constructor in model_dict.items():
#     print(f"Evaluating {name}...")
#     model = constructor.to(device)
    
#     # Load pretrained weights if available
#     weight_path = weights_dict.get(name)
#     if weight_path:
#         checkpoint = torch.load(weight_path, map_location=device,weights_only=False)
#         if 'model' in checkpoint:
#             state_dict = checkpoint['model']
#         else:
#             state_dict = checkpoint
#         model.load_state_dict(state_dict)
    
#     model.eval()
#     # correct = total = 0
#     # with torch.no_grad():
#     #     for imgs, labels in test_loader:
#     #         imgs, labels = imgs.to(device), labels.to(device)
#     #         logits = model(imgs)
#     #         preds = logits.argmax(1)
#     #         correct += (preds == labels).sum().item()
#     #         total += labels.size(0)

#     # test_acc = 100 * correct / total
#     # print(f"🎯 {name} Test Accuracy on CIFAR-100: {test_acc:.2f}%\n")
#     correct1 = correct5 = total = 0
#     true_probs_correct = []
#     true_probs_wrong = []
#     pred_probs_wrong = []   # probability of predicted (wrong) class
    
#     with torch.no_grad():
#         for imgs, labels in test_loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             logits = model(imgs)

#             probs = F.softmax(logits, dim=1)
#             preds1 = logits.argmax(1)

#             # ---- Accuracy ----
#             correct_mask = preds1 == labels
#             correct1 += correct_mask.sum().item()

#             _, top5 = logits.topk(5, dim=1)
#             correct5 += top5.eq(labels.view(-1, 1)).sum().item()

#             # ---- Probabilities ----
#             batch_true_probs = probs[torch.arange(labels.size(0)), labels]
#             batch_pred_probs = probs[torch.arange(labels.size(0)), preds1]

#             # Split into correct/wrong cases
#             true_probs_correct.extend(batch_true_probs[correct_mask].cpu().tolist())
#             true_probs_wrong.extend(batch_true_probs[~correct_mask].cpu().tolist())
#             pred_probs_wrong.extend(batch_pred_probs[~correct_mask].cpu().tolist())

#             total += labels.size(0)

#     top1_acc = 100 * correct1 / total
#     top5_acc = 100 * correct5 / total

#     avg_true_prob_correct = sum(true_probs_correct) / len(true_probs_correct)
#     avg_true_prob_wrong = sum(true_probs_wrong) / len(true_probs_wrong)
#     avg_pred_prob_wrong = sum(pred_probs_wrong) / len(pred_probs_wrong)

#     print(f"🎯 {name} Test Accuracy on CIFAR-100:")
#     print(f"   Top-1 = {top1_acc:.2f}%")
#     print(f"   Top-5 = {top5_acc:.2f}%")
#     print(f"   Avg. prob (true class) when correct = {avg_true_prob_correct:.4f}")
#     print(f"   Avg. prob (true class) when WRONG   = {avg_true_prob_wrong:.4f}")
#     print(f"   Avg. prob (pred class) when WRONG   = {avg_pred_prob_wrong:.4f}\n")




#     wrong_probs_list = []
#     wrong_labels_list = []
#     wrong_preds_list = []
    
#     with torch.no_grad():
#         for imgs, labels in test_loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             logits = model(imgs)
#             probs = F.softmax(logits, dim=1)
    
#             preds1 = logits.argmax(1)
#             wrong_mask = preds1 != labels
    
#             if wrong_mask.any():
#                 wrong_probs_list.append(probs[wrong_mask].cpu())
#                 wrong_labels_list.append(labels[wrong_mask].cpu())
#                 wrong_preds_list.append(preds1[wrong_mask].cpu())
    
#     # Concatenate all batches
#     wrong_probs = torch.cat(wrong_probs_list, dim=0)        # [num_wrong, 100]
#     wrong_labels = torch.cat(wrong_labels_list, dim=0)      # [num_wrong]
#     wrong_preds = torch.cat(wrong_preds_list, dim=0)        # [num_wrong]
    
#     # ---------------------------
#     # Split probabilities
#     # ---------------------------
#     true_class_probs = wrong_probs[torch.arange(wrong_probs.size(0)), wrong_labels]
#     pred_class_probs = wrong_probs[torch.arange(wrong_probs.size(0)), wrong_preds]
    
#     remaining_probs = wrong_probs.clone()
#     remaining_probs[torch.arange(wrong_probs.size(0)).unsqueeze(1), wrong_labels.unsqueeze(1)] = 0
#     remaining_probs[torch.arange(wrong_probs.size(0)).unsqueeze(1), wrong_preds.unsqueeze(1)] = 0
#     remaining_probs_flat = remaining_probs.flatten()

    
#     # ---------------------------
#     # Visualization
#     # ---------------------------
#     plt.figure(figsize=(12,5))
    
#     plt.subplot(1,2,1)
#     plt.hist(true_class_probs.numpy(), bins=50, alpha=0.7, label="True class prob")
#     plt.hist(pred_class_probs.numpy(), bins=50, alpha=0.7, label="Pred class prob")
#     plt.xlabel("Probability")
#     plt.ylabel("Frequency")
#     plt.title("True vs Predicted class probabilities (wrong preds)")
#     plt.legend()
    
#     plt.subplot(1,2,2)
#     plt.hist(remaining_probs_flat.numpy(), bins=50, color='orange')
#     plt.xlabel("Probability")
#     plt.ylabel("Frequency")
#     plt.title("Remaining classes probabilities (wrong preds)")
    
#     plt.show()


# ShuffleNetV2 Architecture

In [11]:
# '''ShuffleNetV2 in PyTorch.
# See the paper "ShuffleNet V2: Practical Guidelines for Efficient CNN Architecture Design" for more details.
# '''

# class ShuffleBlock(nn.Module):
#     def __init__(self, groups=2):
#         super(ShuffleBlock, self).__init__()
#         self.groups = groups

#     def forward(self, x):
#         '''Channel shuffle: [N,C,H,W] -> [N,g,C/g,H,W] -> [N,C/g,g,H,w] -> [N,C,H,W]'''
#         N, C, H, W = x.size()
#         g = self.groups
#         return x.view(N, g, C//g, H, W).permute(0, 2, 1, 3, 4).reshape(N, C, H, W)


# class SplitBlock(nn.Module):
#     def __init__(self, ratio):
#         super(SplitBlock, self).__init__()
#         self.ratio = ratio

#     def forward(self, x):
#         c = int(x.size(1) * self.ratio)
#         return x[:, :c, :, :], x[:, c:, :, :]


# class BasicBlock(nn.Module):
#     def __init__(self, in_channels, split_ratio=0.5, is_last=False):
#         super(BasicBlock, self).__init__()
#         self.is_last = is_last
#         self.split = SplitBlock(split_ratio)
#         in_channels = int(in_channels * split_ratio)
#         self.conv1 = nn.Conv2d(in_channels, in_channels,
#                                kernel_size=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(in_channels)
#         self.conv2 = nn.Conv2d(in_channels, in_channels,
#                                kernel_size=3, stride=1, padding=1, groups=in_channels, bias=False)
#         self.bn2 = nn.BatchNorm2d(in_channels)
#         self.conv3 = nn.Conv2d(in_channels, in_channels,
#                                kernel_size=1, bias=False)
#         self.bn3 = nn.BatchNorm2d(in_channels)
#         self.shuffle = ShuffleBlock()

#     def forward(self, x):
#         x1, x2 = self.split(x)
#         out = F.relu(self.bn1(self.conv1(x2)))
#         out = self.bn2(self.conv2(out))
#         preact = self.bn3(self.conv3(out))
#         out = F.relu(preact)
#         # out = F.relu(self.bn3(self.conv3(out)))
#         preact = torch.cat([x1, preact], 1)
#         out = torch.cat([x1, out], 1)
#         out = self.shuffle(out)
#         if self.is_last:
#             return out, preact
#         else:
#             return out


# class DownBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(DownBlock, self).__init__()
#         mid_channels = out_channels // 2
#         # left
#         self.conv1 = nn.Conv2d(in_channels, in_channels,
#                                kernel_size=3, stride=2, padding=1, groups=in_channels, bias=False)
#         self.bn1 = nn.BatchNorm2d(in_channels)
#         self.conv2 = nn.Conv2d(in_channels, mid_channels,
#                                kernel_size=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(mid_channels)
#         # right
#         self.conv3 = nn.Conv2d(in_channels, mid_channels,
#                                kernel_size=1, bias=False)
#         self.bn3 = nn.BatchNorm2d(mid_channels)
#         self.conv4 = nn.Conv2d(mid_channels, mid_channels,
#                                kernel_size=3, stride=2, padding=1, groups=mid_channels, bias=False)
#         self.bn4 = nn.BatchNorm2d(mid_channels)
#         self.conv5 = nn.Conv2d(mid_channels, mid_channels,
#                                kernel_size=1, bias=False)
#         self.bn5 = nn.BatchNorm2d(mid_channels)

#         self.shuffle = ShuffleBlock()

#     def forward(self, x):
#         # left
#         out1 = self.bn1(self.conv1(x))
#         out1 = F.relu(self.bn2(self.conv2(out1)))
#         # right
#         out2 = F.relu(self.bn3(self.conv3(x)))
#         out2 = self.bn4(self.conv4(out2))
#         out2 = F.relu(self.bn5(self.conv5(out2)))
#         # concat
#         out = torch.cat([out1, out2], 1)
#         out = self.shuffle(out)
#         return out


# class ShuffleNetV2(nn.Module):
#     def __init__(self, net_size, num_classes=100):
#         super(ShuffleNetV2, self).__init__()
#         out_channels = configs[net_size]['out_channels']
#         num_blocks = configs[net_size]['num_blocks']

#         # self.conv1 = nn.Conv2d(3, 24, kernel_size=3,
#         #                        stride=1, padding=1, bias=False)
#         self.conv1 = nn.Conv2d(3, 24, kernel_size=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(24)
#         self.in_channels = 24
#         self.layer1 = self._make_layer(out_channels[0], num_blocks[0])
#         self.layer2 = self._make_layer(out_channels[1], num_blocks[1])
#         self.layer3 = self._make_layer(out_channels[2], num_blocks[2])
#         self.conv2 = nn.Conv2d(out_channels[2], out_channels[3],
#                                kernel_size=1, stride=1, padding=0, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels[3])
#         self.linear = nn.Linear(out_channels[3], num_classes)

#     def _make_layer(self, out_channels, num_blocks):
#         layers = [DownBlock(self.in_channels, out_channels)]
#         for i in range(num_blocks):
#             layers.append(BasicBlock(out_channels, is_last=(i == num_blocks - 1)))
#             self.in_channels = out_channels
#         return nn.Sequential(*layers)

#     def get_feat_modules(self):
#         feat_m = nn.ModuleList([])
#         feat_m.append(self.conv1)
#         feat_m.append(self.bn1)
#         feat_m.append(self.layer1)
#         feat_m.append(self.layer2)
#         feat_m.append(self.layer3)
#         return feat_m

#     def get_bn_before_relu(self):
#         raise NotImplementedError('ShuffleNetV2 currently is not supported for "Overhaul" teacher')

#     def forward(self, x, is_feat=False, preact=False):
#         out = F.relu(self.bn1(self.conv1(x)))
#         # out = F.max_pool2d(out, 3, stride=2, padding=1)
#         f0 = out
#         out, f1_pre = self.layer1(out)
#         f1 = out
#         out, f2_pre = self.layer2(out)
#         f2 = out
#         out, f3_pre = self.layer3(out)
#         f3 = out
#         out = F.relu(self.bn2(self.conv2(out)))
#         # out = F.avg_pool2d(out, 4)
#         '''
#         making it adaptive
#         '''
#         out = F.adaptive_avg_pool2d(out, 1)
#         out = out.view(out.size(0), -1)
#         f4 = out
#         out = self.linear(out)
#         if is_feat:
#             if preact:
#                 return [f0, f1_pre, f2_pre, f3_pre, f4], out
#             else:
#                 return [f0, f1, f2, f3, f4], out
#         else:
#             return out


# configs = {
#     0.2: {
#         'out_channels': (40, 80, 160, 512),
#         'num_blocks': (3, 3, 3)
#     },

#     0.3: {
#         'out_channels': (40, 80, 160, 512),
#         'num_blocks': (3, 7, 3)
#     },

#     0.5: {
#         'out_channels': (48, 96, 192, 1024),
#         'num_blocks': (3, 7, 3)
#     },

#     1: {
#         'out_channels': (116, 232, 464, 1024),
#         'num_blocks': (3, 7, 3)
#     },
#     1.5: {
#         'out_channels': (176, 352, 704, 1024),
#         'num_blocks': (3, 7, 3)
#     },
#     2: {
#         'out_channels': (224, 488, 976, 2048),
#         'num_blocks': (3, 7, 3)
#     }
# }


# def ShuffleV2(**kwargs):
#     model = ShuffleNetV2(net_size=1, **kwargs)
#     return model


# # if __name__ == '__main__':
# #     net = ShuffleV2(num_classes=100)
# #     x = torch.randn(3, 3, 32, 32)
# #     import time
# #     a = time.time()
# #     feats, logit = net(x, is_feat=True, preact=True)
# #     b = time.time()
# #     print(b - a)
# #     for f in feats:
# #         print(f.shape, f.min().item())
# #     print(logit.shape)

# ResNet32x4 to SN-V1

# Building Functions
- Data Loaders
- Test
- Train (CE only)

In [12]:
# def create_curriculum_stages(start_size=24, end_size=32, increment=2, 
#                            epochs_per_stage=48, bridge_percentage=0.1,
#                            alternating_pattern='ABAB'):
#     """
#     Create curriculum with alternating bridging patterns
#     """
#     sizes = list(range(start_size, end_size + increment, increment))
#     stages = []
    
#     for i, size in enumerate(sizes):
#         if i == len(sizes) - 1:  # Last stage - no bridging
#             stages.append((size, epochs_per_stage))
#             continue
        
#         # Calculate bridging epochs
#         bridge_count = int(epochs_per_stage * bridge_percentage)
#         main_count = epochs_per_stage - bridge_count
#         next_size = sizes[i + 1]
        
#         # Add main learning epochs first
#         stages.append((size, main_count))
        
#         # Generate alternating pattern for bridging
#         bridge_sequence = generate_alternating_sequence(
#             main_size=size, 
#             bridge_size=next_size, 
#             total_epochs=bridge_count,
#             pattern=alternating_pattern
#         )
        
#         # Add alternating bridge epochs
#         stages.extend(bridge_sequence)
    
#     return stages

# def generate_alternating_sequence(main_size, bridge_size, total_epochs, pattern='ABAB'):
#     """Generate alternating epoch sequence - total must equal total_epochs"""
#     sequence = []
    
#     if pattern == 'ABAB':
#         for i in range(total_epochs):
#             size = bridge_size if i % 2 == 0 else main_size
#             sequence.append((size, 1))
    
#     elif pattern == 'AABB':
#         # Alternate in pairs, but respect total_epochs limit
#         i = 0
#         while i < total_epochs:
#             # Add A pair (bridge_size)
#             pair_size = min(2, total_epochs - i)
#             for _ in range(pair_size):
#                 sequence.append((bridge_size, 1))
#                 i += 1
#                 if i >= total_epochs:
#                     break
            
#             # Add B pair (main_size) 
#             if i < total_epochs:
#                 pair_size = min(2, total_epochs - i)
#                 for _ in range(pair_size):
#                     sequence.append((main_size, 1))
#                     i += 1
    
#     elif 'A' in pattern and 'B' in pattern:
#         # Parse A2B1 format
#         import re
#         matches = re.findall(r'([AB])(\d+)', pattern)
#         if matches:
#             a_count = int([count for letter, count in matches if letter == 'A'][0])
#             b_count = int([count for letter, count in matches if letter == 'B'][0])
            
#             cycle_length = a_count + b_count
#             full_cycles = total_epochs // cycle_length
#             remainder = total_epochs % cycle_length
            
#             # Add full cycles
#             for _ in range(full_cycles):
#                 for _ in range(a_count):
#                     sequence.append((bridge_size, 1))
#                 for _ in range(b_count):
#                     sequence.append((main_size, 1))
            
#             # Add remainder
#             remaining_a = min(remainder, a_count)
#             remaining_b = max(0, remainder - a_count)
            
#             for _ in range(remaining_a):
#                 sequence.append((bridge_size, 1))
#             for _ in range(remaining_b):
#                 sequence.append((main_size, 1))
    
#     return sequence

# # Example 1: Basic alternating (ABAB pattern)
# stages = create_curriculum_stages(
#     start_size=24, 
#     end_size=32, 
#     increment=2,
#     epochs_per_stage=48, 
#     bridge_percentage=0.1,
#     alternating_pattern='A2B1'
# )

# print("A2B1 Pattern:")
# for stage in stages:
#     print(f"Size {stage[0]}: {stage[1]} epochs")

# # Output:
# # Size 24: 43 epochs  (main learning)
# # Size 26: 1 epochs   (bridge)
# # Size 24: 1 epochs   (back to main)
# # Size 26: 1 epochs   (bridge)
# # Size 24: 1 epochs   (back to main)
# # Size 26: 1 epochs   (bridge)
# # Size 26: 43 epochs  (main learning for 26)
# # Size 28: 1 epochs   (bridge to 28)
# # Size 26: 1 epochs   (back to 26)
# # ... and so on

# # print(f"\nTotal epochs for first curriculum: {sum(epoch for size, epoch in stages[:6])}")  #

In [13]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# # use_dp = torch.cuda.device_count() > 1
# torch.manual_seed(27)
# np.random.seed(27)

# mean = (0.5071, 0.4867, 0.4408)  # CIFAR-100 mean
# std  = (0.2675, 0.2565, 0.2761)  # CIFAR-100 std

# batch_size = 64

# # Precompute DataLoaders for each resolution
# stages = [(r) for r in [(24, 48), (26, 48), (28, 48), (30, 48), (32, 48)]]
# dataloader_dict = {}

# '''
# Copied from the paper as it is.

# they are not using any validation sets. Training it on the entire train set!
#     train_transform = transforms.Compose([
#         transforms.RandomCrop(32, padding=4),
#         transforms.RandomHorizontalFlip(),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=mean, std=stdv),
#     ])
#     test_transform = transforms.Compose([
#         transforms.ToTensor(),
#         transforms.Normalize(mean=mean, std=stdv),
#     ])
    
#     '''

# for resolution, _ in stages:
#     train_tf = transforms.Compose([
#         transforms.RandomCrop(resolution,padding=4),
#         transforms.RandomHorizontalFlip(),
#         # transforms.Resize(32,antialias=True),
#         transforms.ToTensor(),
#         transforms.Normalize(mean, std),
#     ])
    
#     train_set = datasets.CIFAR100('./data', train=True, download=False, transform=train_tf)
    
#     train_loader = DataLoader(train_set, batch_size=batch_size,
#                               shuffle=True, num_workers=0, pin_memory=True)
#     dataloader_dict[resolution] = {
#         'train': train_loader
#     }

# # Test loader (fixed resolution)
# test_tf = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean, std),
# ])

# test_ds = datasets.CIFAR100('./data', train=False, transform=test_tf)
# test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)


# # learning_rate is divided by 10 

# # ---------------------------
# # Test loop
# # ---------------------------

# def test(Test_model):
    
#     Test_model.eval()
#     correct_val = total_val = 0
    
#     with torch.no_grad():
#         for imgs, labels in test_loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             logits = Test_model(imgs)
#             preds = logits.argmax(1)
#             correct_val += (preds == labels).sum().item()
#             total_val += labels.size(0)
#     test_acc = 100 * correct_val / total_val

#     # print(f"Test Acc = {val_acc:.2f}%")
#     return test_acc

# # ---------------------------
# # Training loop
# # ---------------------------

# # def train(model, model_type):
    
# #     # Loss + optimizer
# #     lr = 0.05 # as per the paper
# #     criterion = nn.CrossEntropyLoss()
# #     optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

# #     # Step LR schedule: decay at 150, 180, 210 epochs
# #     scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[150, 180, 210], gamma=0.1)

# #     best_val_acc = 0.0
# #     num_epochs = 240
    
# #     for res, epochs in stages:
        
# #         print(f"\n=== Training at resolution {res}px ===")
# #         tr_loader = dataloader_dict[res]['train']
        
# #         for e in range(1, epochs+1):
            
# #             model.train()
# #             total_loss = 0
# #             correct = total = 0
        
# #             for imgs, labels in tqdm(tr_loader, desc=f"Epoch {e}/{num_epochs}"):
                
# #                 imgs, labels = imgs.to(device), labels.to(device)
# #                 optimizer.zero_grad()
# #                 logits = model(imgs)
# #                 loss = criterion(logits, labels)
# #                 loss.backward()
# #                 optimizer.step()
        
# #                 total_loss += loss.item()
# #                 preds = logits.argmax(1)
# #                 correct += (preds == labels).sum().item()
# #                 total += labels.size(0)
        
# #             train_acc = 100 * correct / total
# #             scheduler.step()

# #             test_acc = test(model)
            
# #             print(f"Epoch {e}: Train Acc = {train_acc:.2f}% Test Accuracy = {test_acc:.2f}%") 
            
# #             # Save best model
# #             if test_acc > best_val_acc:
# #                 best_val_acc = test_acc
# #                 torch.save(model.state_dict(), f"resnet{model_type}_student.pth")
# #                 print(f"→ Saved best model at epoch {e} with Test Acc = {train_acc:.2f}%")
# #     print("✅ Training completed!")

# Building Functions for KD
- KD Loss
- Train Via KD

In [14]:
# def kd_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.9): 
#     """
#     alpha = 0.1 as per the github repo
#     Compute KD loss = α * KD + (1-α) * CE
#     T = temperature
#     α = weight for soft distillation loss
#     """
#     # Hard-label loss
#     ce_loss = F.cross_entropy(student_logits, labels)

#     # Soft distillation loss
#     kd = F.kl_div(
#         F.log_softmax(student_logits / T, dim=1),
#         F.softmax(teacher_logits / T, dim=1),
#         reduction="batchmean"
#     ) * (T * T)

#     return alpha * kd + (1 - alpha) * ce_loss


# def train_via_KD(t_model,s_model, model_type):

#     t_model.eval()
#     for p in t_model.parameters():
#         p.requires_grad = False
        
#     # Loss + optimizer
#     lr = 0.01 # as per the paper
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.SGD(s_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
#     # scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

#     # Step LR schedule: decay at 150, 180, 210 epochs
#     scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[48, 96, 192], gamma=0.1)

#     best_test_acc = 0.0
#     num_epochs = 240
    
#     for res, epochs in stages:
        
            
#         print(f"\n=== Training student at resolution {res}px for {epochs} epochs ===")
#         tr_loader = dataloader_dict[res]['train']
        
#         for e in range(1, epochs+1):

#             s_model.train()
            
#             total_loss = 0
#             correct = total = 0
        
#             for imgs, labels in tqdm(tr_loader, desc=f"Epoch {e}/{num_epochs}"):
                
#                 imgs, labels = imgs.to(device), labels.to(device)
#                 optimizer.zero_grad()

#                 with torch.no_grad():
#                     t_logits = t_model(imgs)
#                     t_preds = t_logits.argmax(1)

#                 mask = (t_preds == labels)
#                 if mask.sum() == 0:  # If teacher got entire batch wrong
#                     continue  # skip batch

#                 imgs = imgs[mask]
#                 labels = labels[mask]
#                 t_logits = t_logits[mask]
                
#                 s_logits = s_model(imgs)
                
#                 loss = kd_loss(s_logits, t_logits, labels)
#                 loss.backward()
#                 optimizer.step()
        
#                 total_loss += loss.item()
#                 preds = s_logits.argmax(1)
#                 correct += (preds == labels).sum().item()
#                 total += imgs.size(0)

#             train_acc = 100 * correct / total
#             scheduler.step()

#             #Validation logic comes here
#             val_acc = test(s_model)
            
#             print(f"Epoch {e}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%")
        
#             # Save best model
#             if val_acc > best_test_acc:
#                 best_test_acc = val_acc
#                 torch.save(model.state_dict(), f"resnet_KD_{model_type}_student.pth")
#                 print(f"→ Saved student best model at epoch {e} with Val Acc = {val_acc:.2f}%")
#     print("✅ Teacher training completed!")

In [15]:
# student_32_vgg8 = vgg8_bn(target_resolution=32, num_classes=100).to(device)
# train(student_32_vgg8, "randomtesting")

In [16]:
# #declare a teacher model and student model , transfer to gpu and train.

# for name, constructor in model_dict.items():
#     if name == "ResNet32x4":
#         print(f"Loading {name}...")
#         t_model = constructor.to(device)

#         # Load pretrained weights if available
#         weight_path = weights_dict.get(name)
#         if weight_path:
#             checkpoint = torch.load(weight_path, map_location=device,weights_only=False)
#             if 'model' in checkpoint:
#                 state_dict = checkpoint['model']
#             else:
#                 state_dict = checkpoint
#             t_model.load_state_dict(state_dict)

# s_model = ShuffleV1().to(device)

# # print(t_model)
# # print(s_model)
# train_via_KD(t_model,s_model, "Shufflev1")

# Part 1

In [17]:
# kd_crop_curriculum_refactor_imagenet.py
import gc
import io
import json
import math
import os
import random
from bisect import bisect_right
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from datasets import DatasetDict, load_dataset, load_from_disk
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


def extract_logits(model_out: Any) -> torch.Tensor:
    if isinstance(model_out, (tuple, list)):
        return model_out[0]
    return model_out


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct = total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = extract_logits(model(x))
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    model.train()
    return 100.0 * correct / max(1, total)


# ---------------------------- Configs ---------------------------------
@dataclass
class DataConfig:
    root: str = "./data"
    cache_dir: str = "./.cache"
    mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    std: Tuple[float, float, float] = (0.229, 0.224, 0.225)
    batch_size: int = 16
    eval_batch_size: int = 64
    num_workers: int = 4
    persistent_workers: bool = True
    pin_memory: bool = True
    image_size: int = 224
    resize_size: int = 256
    train_split: str = "train"
    val_split: str = "validation"
    # (resolution, epochs_at_resolution)
    stages: Tuple[Tuple[int, int], ...] = ((96, 48), (128, 48), (160, 48), (192, 48), (224, 48))


@dataclass
class KDConfig:
    T: float = 4.0
    alpha: float = 0.9


@dataclass
class OptimConfig:
    base_lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 1e-4
    lr_half_on_stage_change: bool = True
    milestones: Tuple[int, int, int] = (72, 144, 192)
    milestone_gamma: float = 0.1


@dataclass
class TrainConfig:
    epochs: int = 240
    grad_accum_steps: int = 16
    seed: int = 27
    save_path: str = "./.cache/checkpoints/student_r18_kd_imagenet.pth"


@dataclass
class TGSCConfig:
    precompute_batch_size: int = 64
    precompute_workers: int = 2
    use_gt_target: bool = True
    w_cam: float = 0.7
    w_sal: float = 0.3
    temp: float = 1.5
    gamma: float = 1.2
    pct_lo: float = 1.0
    pct_hi: float = 99.0
    stride: int = 8
    k_per_img: int = 6
    min_frac: float = 0.20
    min_mean: float = 0.20
    eval_chunk: int = 128
    cache: bool = True
    force_rebuild: bool = False
    flip_prob: float = 0.5
    aug_padding: int = 2
    top_k_crops: int = 2
    low_k_crops: int = 1
    score_high: float = 0.75
    score_low: float = 0.35
    teacher_cache_shard_size: int = 128
    teacher_cache_batch_size: int = 64
    teacher_cache_min_batch_size: int = 8
    crop_cache_shard_size: int = 4096


# ----------------------- Filesystem / Dataset helpers ------------------
def _read_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _write_json(path: Path, payload: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def resolve_imagenet_root(preferred_root: str = "./data") -> Path:
    candidates = [Path("/data"), Path(preferred_root), Path("./data")]
    for root in candidates:
        parquet_dir = root / "data"
        if parquet_dir.exists() and any(parquet_dir.glob("train-*.parquet")):
            return root
    raise FileNotFoundError(
        "Could not find ImageNet parquet shards. Expected files like '<root>/data/train-*.parquet'."
    )


def ensure_cache_layout(cache_root: str | Path) -> Dict[str, Path]:
    base = Path(cache_root)
    paths = {
        "base": base,
        "imagenet_ds": base / "imagenet_ds",
        "teacher_union": base / "teacher_union",
        "tgsc_crops": base / "tgsc_crops",
        "checkpoints": base / "checkpoints",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def load_or_build_imagenet_ds(data_cfg: DataConfig) -> DatasetDict:
    cache_paths = ensure_cache_layout(data_cfg.cache_dir)
    root = Path(data_cfg.root)
    parquet_dir = root / "data"
    if not parquet_dir.exists() or not any(parquet_dir.glob("train-*.parquet")):
        root = resolve_imagenet_root(data_cfg.root)
        parquet_dir = root / "data"

    save_dir = cache_paths["imagenet_ds"] / "saved_ds"
    hf_cache = cache_paths["imagenet_ds"] / "hf_cache"

    if save_dir.exists():
        print(f"[DATA] Loading cached ImageNet DatasetDict from {save_dir}")
        return load_from_disk(str(save_dir), keep_in_memory=False)

    train_files = sorted(str(p) for p in parquet_dir.glob("train-*.parquet"))
    val_files = sorted(str(p) for p in parquet_dir.glob("validation-*.parquet"))
    test_files = sorted(str(p) for p in parquet_dir.glob("test-*.parquet"))

    if not train_files:
        raise RuntimeError(f"No train parquet files found under: {parquet_dir}")

    data_files: Dict[str, List[str]] = {"train": train_files}
    if val_files:
        data_files["validation"] = val_files
    elif test_files:
        data_files["validation"] = test_files
    else:
        raise RuntimeError("No validation/test parquet files found for ImageNet evaluation split.")

    print(f"[DATA] Building ImageNet DatasetDict from parquet shards in: {parquet_dir}")
    ds = load_dataset(
        "parquet",
        data_files=data_files,
        cache_dir=str(hf_cache),
        keep_in_memory=False,
    )
    ds.save_to_disk(str(save_dir))
    print(f"[DATA] Saved ImageNet DatasetDict to: {save_dir}")
    return load_from_disk(str(save_dir), keep_in_memory=False)


class ImageNetTorchDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def _to_pil(self, image_obj):
        if isinstance(image_obj, Image.Image):
            img = image_obj
        elif isinstance(image_obj, dict):
            if image_obj.get("bytes") is not None:
                img = Image.open(io.BytesIO(image_obj["bytes"]))
            elif image_obj.get("path"):
                img = Image.open(image_obj["path"])
            else:
                raise ValueError("Unsupported image dict format in dataset row.")
        else:
            img = Image.fromarray(np.array(image_obj))
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

    def __getitem__(self, idx: int):
        row = self.hf_split[idx]
        img = self._to_pil(row["image"])
        x = self.transform(img) if self.transform is not None else transforms.ToTensor()(img)
        y = int(row["label"])
        return x, y


def make_val_loader(
    imagenet_ds: DatasetDict,
    data_cfg: DataConfig,
) -> DataLoader:
    val_split = data_cfg.val_split if data_cfg.val_split in imagenet_ds else (
        "validation" if "validation" in imagenet_ds else "test"
    )
    if val_split not in imagenet_ds:
        raise RuntimeError(f"Validation split not found. Available: {list(imagenet_ds.keys())}")

    val_tfm = transforms.Compose([
        transforms.Resize(data_cfg.resize_size),
        transforms.CenterCrop(data_cfg.image_size),
        transforms.ToTensor(),
        transforms.Normalize(data_cfg.mean, data_cfg.std),
    ])
    val_ds = ImageNetTorchDataset(imagenet_ds[val_split], transform=val_tfm)
    return DataLoader(
        val_ds,
        batch_size=data_cfg.eval_batch_size,
        shuffle=False,
        num_workers=data_cfg.num_workers,
        pin_memory=data_cfg.pin_memory,
    )


def build_stage_plan(stages: List[Tuple[int, int]]) -> List[int]:
    per_epoch_res: List[int] = []
    for res, n in stages:
        per_epoch_res.extend([res] * n)
    return per_epoch_res


# ---------------------------- Model helpers ----------------------------
def pick_target_layer(model: nn.Module) -> nn.Module:
    base = model.module if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)) else model

    if hasattr(base, "layer4") and isinstance(base.layer4, nn.Sequential):
        last_block = base.layer4[-1]
        if hasattr(last_block, "conv3"):
            return last_block.conv3
        if hasattr(last_block, "conv2"):
            return last_block.conv2
        return last_block

    if hasattr(base, "features") and isinstance(base.features, nn.Sequential):
        for layer in reversed(base.features):
            if isinstance(layer, nn.Conv2d):
                return layer
        return base.features[-1]

    last_conv = None
    for name, module in base.named_modules():
        if isinstance(module, nn.Conv2d) and "classifier" not in name.lower() and "fc" not in name.lower():
            last_conv = module

    if last_conv is not None:
        return last_conv

    raise ValueError("Could not infer Grad-CAM target layer.")


def infer_num_classes(model: nn.Module, device: Optional[torch.device] = None) -> int:
    base = model.module if isinstance(model, nn.DataParallel) else model
    for attr in ("fc", "classifier", "head"):
        layer = getattr(base, attr, None)
        if isinstance(layer, nn.Linear):
            return layer.out_features
    with torch.no_grad():
        dev = device or next(base.parameters()).device
        dummy = torch.zeros(1, 3, 224, 224, device=dev)
        logits = extract_logits(base(dummy))
    return logits.shape[1]


# ---------------------------- KD loss ----------------------------------
class KDLoss(nn.Module):
    def __init__(self, T: float = 4.0, alpha: float = 0.9):
        super().__init__()
        self.T = T
        self.alpha = alpha

    def forward(
        self,
        s_logits: torch.Tensor,
        t_logits: torch.Tensor,
        y: torch.Tensor,
        *,
        weights: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        T, a = self.T, self.alpha
        ce_each = F.cross_entropy(s_logits, y, reduction="none")
        log_s = F.log_softmax(s_logits / T, dim=1)
        p_t = F.softmax(t_logits.detach() / T, dim=1)
        log_t = torch.log(p_t + 1e-8)
        kd_each = (p_t * (log_t - log_s)).sum(dim=1) * (T * T)
        if weights is not None:
            kd_each = kd_each * weights
        ce = ce_each.mean()
        kd = kd_each.mean()
        loss = (1.0 - a) * ce + a * kd
        return {
            "loss": loss,
            "ce": ce.detach(),
            "kd": kd.detach(),
            "ce_each": ce_each.detach(),
            "kd_each": kd_each.detach(),
        }


class ExtraLosses:
    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        return {"extra_loss": torch.tensor(0.0, device=kwargs["device"])}


# ------------------------- TGSC internals ------------------------------
class _LayerTaps:
    def __init__(self, target_layer: nn.Module, *, capture_grad: bool = True):
        self.A = None
        self.dY = None
        self._h1 = target_layer.register_forward_hook(self._store_act)
        self._h2 = target_layer.register_full_backward_hook(self._store_grad) if capture_grad else None

    def _store_act(self, _module, _inp, out):
        self.A = out

    def _store_grad(self, _module, _gin, gout):
        self.dY = gout[0]

    def close(self):
        self._h1.remove()
        if self._h2 is not None:
            self._h2.remove()


def _gradcam_pp_from_taps(A: torch.Tensor, dY: torch.Tensor) -> torch.Tensor:
    eps = 1e-8
    dY2 = dY * dY
    dY3 = dY2 * dY
    sumA = A.sum(dim=(2, 3), keepdim=True)
    alpha = dY2 / (2.0 * dY2 + sumA * dY3.sum(dim=(2, 3), keepdim=True) + eps)
    w = (alpha * dY.clamp_min(0)).sum(dim=(2, 3), keepdim=True)
    cam = (w * A).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    return cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)


def _soft_or_t(cam: torch.Tensor, sal: torch.Tensor, *, w_cam: float, w_sal: float, temp: float) -> torch.Tensor:
    cam = cam.clamp(0, 1)
    sal = sal.clamp(0, 1)
    if temp != 1.0:
        cam = cam.clamp(1e-8, 1 - 1e-8).pow(1.0 / temp)
        sal = sal.clamp(1e-8, 1 - 1e-8).pow(1.0 / temp)
    return 1.0 - (1.0 - cam).pow(w_cam) * (1.0 - sal).pow(w_sal)


def _percentile_scale_t(x: torch.Tensor, *, lo: float, hi: float) -> torch.Tensor:
    B = x.size(0)
    flat = x.view(B, -1)
    q = torch.tensor([lo / 100.0, hi / 100.0], device=x.device, dtype=x.dtype)
    vals = torch.quantile(flat, q, dim=1)
    lo_v = vals[0].view(B, 1, 1, 1)
    hi_v = vals[1].view(B, 1, 1, 1)
    denom = (hi_v - lo_v).clamp_min(1e-8)
    return ((x - lo_v).clamp_min(0.0) / denom).clamp(0.0, 1.0)


def _build_windows_by_pool(union: torch.Tensor, crop_h: int, crop_w: int, stride: int) -> Tuple[torch.Tensor, torch.Tensor]:
    thr = _percentile_scale_t(union, lo=75, hi=75)
    mask = (union >= thr).float()
    mean_union_map = F.avg_pool2d(union, kernel_size=(crop_h, crop_w), stride=stride, padding=0)
    frac_above_map = F.avg_pool2d(mask, kernel_size=(crop_h, crop_w), stride=stride, padding=0)
    return mean_union_map, frac_above_map


def _gather_top_windows(
    mean_union_map: torch.Tensor,
    frac_map: torch.Tensor,
    *,
    k_per_img: int,
    min_frac: float,
    min_mean: float,
):
    B, _, Hh, Ww = mean_union_map.shape
    flat_score = mean_union_map.view(B, -1)
    flat_frac = frac_map.view(B, -1)
    valid = (flat_frac >= min_frac) & (flat_score >= min_mean)
    score_masked = torch.where(valid, flat_score, torch.full_like(flat_score, -1.0))
    topk = min(k_per_img, Hh * Ww)
    vals, idxs = torch.topk(score_masked, k=topk, dim=1)
    yh = idxs // Ww
    xw = idxs % Ww
    return yh, xw, vals


def _boxes_from_grid(yh: torch.Tensor, xw: torch.Tensor, crop_h: int, crop_w: int, stride: int, H: int, W: int) -> torch.Tensor:
    y0 = yh * stride
    x0 = xw * stride
    y1 = (y0 + crop_h).clamp(max=H)
    x1 = (x0 + crop_w).clamp(max=W)
    return torch.stack([y0, x0, y1, x1], dim=-1)


def _top2_batch(p: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    vals, idx = torch.sort(p, dim=-1, descending=True)
    return idx[:, 0], vals[:, 0], idx[:, 1], vals[:, 1]


def _kl_div_batch(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    eps = 1e-8
    p_ = p.clamp_min(eps)
    q_ = q.clamp_min(eps)
    return (p_ * (p_.log() - q_.log())).sum(dim=1)


def _score_candidates_with_teacher(
    model: nn.Module,
    full_logits: torch.Tensor,
    class_idx: torch.Tensor,
    x: torch.Tensor,
    boxes: torch.Tensor,
    *,
    eval_chunk: int,
):
    device = x.device
    B, K, _ = boxes.shape
    if K == 0:
        return (
            boxes[:, :0],
            torch.empty((B, 0), device=device),
            torch.empty((B, 0, full_logits.size(1)), device=device),
        )

    p_full = torch.softmax(full_logits, dim=-1)
    _, pc1_f, _, pc2_f = _top2_batch(p_full)

    crops = []
    owners = []
    ks = []
    for b in range(B):
        for k in range(K):
            y0, x0, y1, x1 = boxes[b, k].tolist()
            crop = x[b:b + 1, :, y0:y1, x0:x1]
            crops.append(crop)
            owners.append(b)
            ks.append(k)

    if not crops:
        return boxes[:, 0], torch.zeros(B, device=device), p_full

    crops = torch.cat(crops, dim=0)
    preds = []
    with torch.no_grad():
        for i in range(0, crops.size(0), eval_chunk):
            logits_k = extract_logits(model(crops[i:i + eval_chunk]))
            preds.append(torch.softmax(logits_k, dim=-1))
    p_k_all = torch.cat(preds, dim=0)

    scores = torch.empty(p_k_all.size(0), device=device)
    conf_ratio = torch.empty_like(scores)
    kl_norm = torch.empty_like(scores)
    margin_rat = torch.empty_like(scores)

    alpha = math.log(p_k_all.size(1))
    owners_t = torch.tensor(owners, device=device, dtype=torch.long)
    ks_t = torch.tensor(ks, device=device, dtype=torch.long)

    for idx_flat in range(p_k_all.size(0)):
        b = int(owners_t[idx_flat].item())
        pk = p_k_all[idx_flat]
        pf = p_full[b]
        c = int(class_idx[b].item())

        pc_k = pk[c]
        pc_f = pf[c]
        conf_ratio[idx_flat] = pc_k / (pc_f + 1e-8)

        kl = _kl_div_batch(pf.unsqueeze(0), pk.unsqueeze(0))[0]
        kl_norm[idx_flat] = 1.0 - torch.clamp(kl / alpha, max=1.0)

        vals_k, _ = torch.sort(pk, descending=True)
        pc1_k, pc2_k = vals_k[0], vals_k[1]
        margin_rat[idx_flat] = (pc1_k - pc2_k) / (pc1_f[b] - pc2_f[b] + 1e-8)

    scores[:] = (conf_ratio + kl_norm + margin_rat) / 3.0

    score_matrix = torch.full((B, K), -1.0, device=device)
    prob_matrix = torch.zeros((B, K, p_k_all.size(1)), dtype=p_k_all.dtype, device=device)
    for i in range(scores.numel()):
        b = int(owners_t[i].item())
        k = int(ks_t[i].item())
        score_matrix[b, k] = scores[i]
        prob_matrix[b, k] = p_k_all[i]

    sorted_scores, order = torch.sort(score_matrix, dim=1, descending=True)
    order_boxes = order.unsqueeze(-1).expand(-1, -1, 4)
    boxes_sorted = boxes.gather(1, order_boxes)
    order_probs = order.unsqueeze(-1).expand(-1, -1, prob_matrix.size(-1))
    probs_sorted = prob_matrix.gather(1, order_probs)
    return boxes_sorted, sorted_scores, probs_sorted


@dataclass
class TeacherUnionCache:
    cache_dir: Path
    manifest: Dict[str, Any]

    @property
    def num_samples(self) -> int:
        return int(self.manifest["num_samples"])

    @classmethod
    def load(cls, cache_dir: Path) -> "TeacherUnionCache":
        manifest = _read_json(cache_dir / "manifest.json")
        return cls(cache_dir=cache_dir, manifest=manifest)

    def get_batch(self, start_idx: int, end_idx: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        if not (0 <= start_idx < end_idx <= self.num_samples):
            raise IndexError(f"Invalid cache range [{start_idx}, {end_idx}) for N={self.num_samples}")

        unions = []
        logits = []
        for shard in self.manifest["shards"]:
            s0 = int(shard["start"])
            s1 = int(shard["end"])
            if s1 <= start_idx or s0 >= end_idx:
                continue
            payload = torch.load(self.cache_dir / shard["file"], map_location="cpu", weights_only=False)
            lo = max(start_idx, s0) - s0
            hi = min(end_idx, s1) - s0
            unions.append(payload["union_maps"][lo:hi])
            logits.append(payload["logits"][lo:hi])
            del payload

        if not unions:
            raise RuntimeError(f"Teacher cache had no shards for range [{start_idx}, {end_idx})")

        union_t = torch.cat(unions, dim=0).to(device, non_blocking=True)
        logits_t = torch.cat(logits, dim=0).to(device, non_blocking=True)
        return union_t, logits_t


def load_or_build_teacher_union_cache(
    base_loader: DataLoader,
    teacher: nn.Module,
    target_layer: nn.Module,
    *,
    cache_dir: Path,
    shard_size: int,
    teacher_cache_batch_size: int,
    teacher_cache_min_batch_size: int,
    force_rebuild: bool,
    use_gt_target: bool,
    w_cam: float,
    w_sal: float,
    temp: float,
    gamma: float,
    pct_lo: float,
    pct_hi: float,
) -> TeacherUnionCache:
    del shard_size  # Keep interface stable; teacher shards are written per compute chunk.

    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / "manifest.json"

    if manifest_path.exists() and not force_rebuild:
        print(f"[TGSC] Reusing teacher-union cache from {cache_dir}")
        return TeacherUnionCache.load(cache_dir)

    for f in cache_dir.glob("shard_*.pt"):
        f.unlink()
    if manifest_path.exists():
        manifest_path.unlink()

    device = next(teacher.parameters()).device
    teacher.eval()
    teacher_params = list(teacher.parameters())
    teacher_req_grad = [p.requires_grad for p in teacher_params]
    for p in teacher_params:
        p.requires_grad_(False)

    target_chunk = max(1, int(teacher_cache_batch_size))
    min_chunk = max(1, int(teacher_cache_min_batch_size))
    if target_chunk < min_chunk:
        target_chunk = min_chunk

    total = 0
    shards: List[Dict[str, Any]] = []

    def _compute_chunk(x_chunk: torch.Tensor, y_chunk: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        xb = x_chunk.to(device, non_blocking=True).requires_grad_(True)
        yb = y_chunk.to(device, non_blocking=True)
        _, _, H, W = xb.shape

        taps = _LayerTaps(target_layer, capture_grad=False)
        try:
            teacher.zero_grad(set_to_none=True)

            logits = extract_logits(teacher(xb))
            probs_full = torch.softmax(logits.detach(), dim=-1)
            top1 = probs_full.argmax(dim=1)
            target_idx = yb if use_gt_target else top1
            A = taps.A
            if A is None:
                raise RuntimeError("Target layer hook did not capture activations.")

            oh = torch.zeros_like(logits)
            oh.scatter_(1, target_idx.view(-1, 1), 1.0)
            g_x, dY = torch.autograd.grad(
                outputs=logits,
                inputs=[xb, A],
                grad_outputs=oh,
                retain_graph=False,
                create_graph=False,
                allow_unused=False,
            )

            g = g_x.detach().abs().amax(dim=1, keepdim=True)
            sal = (g - g.amin(dim=(2, 3), keepdim=True)) / (
                g.amax(dim=(2, 3), keepdim=True) - g.amin(dim=(2, 3), keepdim=True) + 1e-8
            )

            cam_small = _gradcam_pp_from_taps(A, dY)
            cam = F.interpolate(cam_small, size=(H, W), mode="bilinear", align_corners=False)
            cam = _percentile_scale_t(cam, lo=pct_lo, hi=pct_hi)
            sal = _percentile_scale_t(sal, lo=pct_lo, hi=pct_hi)

            union = _soft_or_t(cam, sal, w_cam=w_cam, w_sal=w_sal, temp=temp)
            if gamma != 1.0:
                union = union.clamp(0, 1).pow(1.0 / max(gamma, 1e-6))

            out_union = union.detach().cpu()
            out_logits = logits.detach().cpu()
            del probs_full, top1, target_idx, A, oh, g_x, dY, g, sal, cam_small, cam, union
            return out_union, out_logits
        finally:
            taps.close()
            taps.A = None
            taps.dY = None
            teacher.zero_grad(set_to_none=True)
            if xb.grad is not None:
                xb.grad = None
            del xb, yb
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, "ipc_collect"):
                    torch.cuda.ipc_collect()

    try:
        print("[TGSC] Building teacher-union cache to disk (adaptive 64-sample chunks)...")
        current_target_chunk = target_chunk
        for xb_batch, yb_batch in tqdm(base_loader, desc="Teacher cache", leave=False):
            batch_size = int(xb_batch.size(0))
            offset = 0

            while offset < batch_size:
                remaining = batch_size - offset
                chunk_floor = max(1, min(min_chunk, remaining))
                chunk_size = min(current_target_chunk, remaining)

                while True:
                    try:
                        x_chunk = xb_batch[offset: offset + chunk_size]
                        y_chunk = yb_batch[offset: offset + chunk_size]
                        union_chunk, logits_chunk = _compute_chunk(x_chunk, y_chunk)

                        shard_name = f"shard_{len(shards):06d}.pt"
                        torch.save(
                            {"union_maps": union_chunk, "logits": logits_chunk},
                            cache_dir / shard_name,
                        )

                        n = int(union_chunk.size(0))
                        shards.append({"file": shard_name, "start": total, "end": total + n})
                        total += n
                        offset += n

                        del x_chunk, y_chunk, union_chunk, logits_chunk
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                            if hasattr(torch.cuda, "ipc_collect"):
                                torch.cuda.ipc_collect()
                        break
                    except torch.cuda.OutOfMemoryError:
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                            if hasattr(torch.cuda, "ipc_collect"):
                                torch.cuda.ipc_collect()
                        gc.collect()
                        if chunk_size <= chunk_floor:
                            raise RuntimeError(
                                "CUDA OOM while building teacher cache; "
                                f"failed at minimum chunk size {chunk_floor}."
                            )
                        next_chunk = max(chunk_floor, chunk_size // 2)
                        print(f"[TGSC] OOM on teacher cache chunk size {chunk_size}; retrying with {next_chunk}.")
                        current_target_chunk = min(current_target_chunk, next_chunk)
                        chunk_size = next_chunk

            del xb_batch, yb_batch
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, "ipc_collect"):
                    torch.cuda.ipc_collect()

        if total == 0:
            raise RuntimeError("Teacher cache build produced zero samples.")

        manifest = {
            "num_samples": total,
            "shards": shards,
            "version": 1,
        }
        _write_json(manifest_path, manifest)
        print(f"[TGSC] Teacher cache ready: {total} samples across {len(shards)} shard(s) at {cache_dir}")
        return TeacherUnionCache(cache_dir=cache_dir, manifest=manifest)
    finally:
        for p, req in zip(teacher_params, teacher_req_grad):
            p.requires_grad_(req)


class DiskCropTensorDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir: Path, *, base_dataset: ImageNetTorchDataset):
        self.cache_dir = Path(cache_dir)
        self.manifest = _read_json(self.cache_dir / "manifest.json")
        self.base_dataset = base_dataset

        version = int(self.manifest.get("version", 0))
        fmt = self.manifest.get("format")
        if version != 2 or fmt != "coords_v2":
            raise RuntimeError(
                f"Unsupported TGSC crop cache format (version={version}, format={fmt}). "
                "Expected version=2, format='coords_v2'. Set force_rebuild=True to regenerate cache."
            )

        resolution = self.manifest.get("resolution")
        if not isinstance(resolution, list) or len(resolution) != 2:
            raise RuntimeError("Invalid TGSC crop cache manifest: 'resolution' must be [height, width].")
        self.crop_h = int(resolution[0])
        self.crop_w = int(resolution[1])

        self.shards = self.manifest["shards"]
        self._ends = [int(s["end"]) for s in self.shards]
        self.transform = None
        self._loaded_shard_idx = -1
        self._loaded_payload = None

    def __len__(self) -> int:
        return int(self.manifest["num_samples"])

    def _resolve_shard_idx(self, idx: int) -> int:
        shard_idx = bisect_right(self._ends, idx)
        if shard_idx >= len(self.shards):
            raise IndexError(idx)
        return shard_idx

    def _load_shard(self, shard_idx: int):
        if self._loaded_shard_idx == shard_idx and self._loaded_payload is not None:
            return
        shard_file = self.cache_dir / self.shards[shard_idx]["file"]
        self._loaded_payload = torch.load(shard_file, map_location="cpu", weights_only=False)
        self._loaded_shard_idx = shard_idx

    def __getitem__(self, idx: int):
        if idx < 0 or idx >= len(self):
            raise IndexError(idx)
        shard_idx = self._resolve_shard_idx(idx)
        shard_meta = self.shards[shard_idx]
        self._load_shard(shard_idx)
        local_idx = idx - int(shard_meta["start"])

        source_idx = int(self._loaded_payload["source_indices"][local_idx].item())
        box = self._loaded_payload["boxes"][local_idx].long()
        y0, x0, y1, x1 = [int(v) for v in box.tolist()]

        x_full, _ = self.base_dataset[source_idx]
        x = x_full[:, y0:y1, x0:x1]
        hh, ww = x.shape[-2:]
        if hh != self.crop_h or ww != self.crop_w:
            pad_t = max(0, self.crop_h - hh)
            pad_l = max(0, self.crop_w - ww)
            x = F.pad(x, (0, pad_l, 0, pad_t), mode="reflect")[:, :self.crop_h, :self.crop_w]

        y = self._loaded_payload["labels"][local_idx]
        s = self._loaded_payload["scores"][local_idx]
        q = self._loaded_payload["quality_flags"][local_idx]

        if self.transform is not None:
            x = self.transform(x.clone())

        return x, y.long(), s.float(), q.long(), idx

    def set_transform(self, transform):
        self.transform = transform


def build_bestcrop_dataset_batchwise(
    base_loader: DataLoader,
    teacher: nn.Module,
    target_layer: nn.Module,
    *,
    crop_size: int | Tuple[int, int],
    cache_root: Path,
    base_dataset: ImageNetTorchDataset,
    crop_shard_size: int,
    force_rebuild: bool,
    use_gt_target: bool,
    w_cam: float,
    w_sal: float,
    temp: float,
    gamma: float,
    pct_lo: float,
    pct_hi: float,
    stride: int,
    k_per_img: int,
    min_frac: float,
    min_mean: float,
    eval_chunk: int,
    top_k_crops: int,
    low_k_crops: int,
    teacher_cache: Optional[TeacherUnionCache] = None,
) -> DiskCropTensorDataset:
    device = next(teacher.parameters()).device
    teacher.eval()

    if isinstance(crop_size, int):
        ch = cw = int(crop_size)
    else:
        ch, cw = int(crop_size[0]), int(crop_size[1])

    res_dir = cache_root / f"res_{ch}x{cw}"
    res_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = res_dir / "manifest.json"

    if manifest_path.exists() and not force_rebuild:
        manifest = _read_json(manifest_path)
        version = int(manifest.get("version", 0))
        fmt = manifest.get("format")
        if version != 2 or fmt != "coords_v2":
            raise RuntimeError(
                f"Existing cache at {res_dir} is not V2 coords format "
                f"(version={version}, format={fmt}). Set force_rebuild=True to regenerate."
            )
        print(f"[TGSC] Reusing disk crop cache for {ch}x{cw} at {res_dir}")
        return DiskCropTensorDataset(res_dir, base_dataset=base_dataset)

    for f in res_dir.glob("shard_*.pt"):
        f.unlink()
    if manifest_path.exists():
        manifest_path.unlink()

    source_idx_buf: List[torch.Tensor] = []
    box_buf: List[torch.Tensor] = []
    label_buf: List[torch.Tensor] = []
    score_buf: List[torch.Tensor] = []
    quality_buf: List[torch.Tensor] = []
    buffered = 0
    total = 0
    shards: List[Dict[str, Any]] = []

    def flush_buffers():
        nonlocal source_idx_buf, box_buf, label_buf, score_buf, quality_buf, buffered, total, shards
        if buffered == 0:
            return
        source_indices = torch.cat(source_idx_buf, dim=0).long()
        boxes = torch.cat(box_buf, dim=0).long()
        labels = torch.cat(label_buf, dim=0).long()
        scores = torch.cat(score_buf, dim=0).float().clamp(0.0, 1.0)
        qflags = torch.cat(quality_buf, dim=0).long()
        shard_name = f"shard_{len(shards):06d}.pt"
        torch.save(
            {
                "source_indices": source_indices,
                "boxes": boxes,
                "labels": labels,
                "scores": scores,
                "quality_flags": qflags,
            },
            res_dir / shard_name,
        )
        n = int(source_indices.size(0))
        shards.append({"file": shard_name, "start": total, "end": total + n})
        total += n
        source_idx_buf = []
        box_buf = []
        label_buf = []
        score_buf = []
        quality_buf = []
        buffered = 0
        del source_indices, boxes, labels, scores, qflags
        gc.collect()

    progress_desc = f"TGSC precompute {ch}x{cw}"
    seen = 0

    for batch_idx, (xb, yb) in enumerate(tqdm(base_loader, desc=progress_desc, leave=False)):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        B, _, H, W = xb.shape
        start_idx = seen
        end_idx = seen + B
        seen += B

        if teacher_cache is not None:
            union, logits = teacher_cache.get_batch(start_idx, end_idx, device)
            probs_full = torch.softmax(logits, dim=-1)
            top1 = probs_full.argmax(dim=1)
            target_idx = yb if use_gt_target else top1
        else:
            xb = xb.requires_grad_(True)
            taps = _LayerTaps(target_layer)
            teacher.zero_grad(set_to_none=True)

            logits = extract_logits(teacher(xb))
            probs_full = torch.softmax(logits.detach(), dim=-1)
            top1 = probs_full.argmax(dim=1)
            target_idx = yb if use_gt_target else top1

            oh = torch.zeros_like(logits)
            oh.scatter_(1, target_idx.view(-1, 1), 1.0)
            logits.backward(gradient=oh, retain_graph=False)

            g = xb.grad.detach().abs().amax(dim=1, keepdim=True)
            sal = (g - g.amin(dim=(2, 3), keepdim=True)) / (
                g.amax(dim=(2, 3), keepdim=True) - g.amin(dim=(2, 3), keepdim=True) + 1e-8
            )

            A, dY = taps.A, taps.dY
            if A is None or dY is None:
                taps.close()
                raise RuntimeError("Target layer hooks did not capture activations or gradients.")

            cam_small = _gradcam_pp_from_taps(A, dY)
            cam = F.interpolate(cam_small, size=(H, W), mode="bilinear", align_corners=False)
            cam = _percentile_scale_t(cam, lo=pct_lo, hi=pct_hi)
            sal = _percentile_scale_t(sal, lo=pct_lo, hi=pct_hi)
            union = _soft_or_t(cam, sal, w_cam=w_cam, w_sal=w_sal, temp=temp)
            if gamma != 1.0:
                union = union.clamp(0, 1).pow(1.0 / max(gamma, 1e-6))

            taps.close()
            teacher.zero_grad(set_to_none=True)
            xb.grad = None
            del oh, g, sal, A, dY, cam_small, cam

        mean_union_map, frac_map = _build_windows_by_pool(union, ch, cw, stride)
        yh, xw, _ = _gather_top_windows(
            mean_union_map,
            frac_map,
            k_per_img=k_per_img,
            min_frac=min_frac,
            min_mean=min_mean,
        )
        boxes = _boxes_from_grid(yh, xw, ch, cw, stride, H, W)
        boxes_sorted, scores_sorted, _ = _score_candidates_with_teacher(
            teacher,
            logits.detach(),
            target_idx,
            xb,
            boxes,
            eval_chunk=eval_chunk,
        )

        for i in range(B):
            sample_scores = scores_sorted[i]
            valid_idx = torch.nonzero(sample_scores >= 0, as_tuple=False).squeeze(-1)
            if valid_idx.numel() == 0:
                continue

            valid_idx = valid_idx.tolist()
            top_sel = valid_idx[:max(0, min(top_k_crops, len(valid_idx)))]
            low_sel_candidates = valid_idx[-max(0, min(low_k_crops, len(valid_idx))):] if low_k_crops > 0 else []
            used = set(top_sel)
            low_sel = []
            for cand in reversed(low_sel_candidates):
                if cand not in used:
                    low_sel.append(cand)
                    used.add(cand)

            selections = [(idx_sel, rank, 1) for rank, idx_sel in enumerate(top_sel)]
            selections.extend([(idx_sel, rank, -1) for rank, idx_sel in enumerate(low_sel)])

            for candidate_idx, rank, quality_flag in selections:
                score_val = sample_scores[candidate_idx]
                if score_val < 0:
                    continue
                y0, x0, y1, x1 = boxes_sorted[i, candidate_idx].long().tolist()
                source_idx_buf.append(torch.tensor([start_idx + i], dtype=torch.long))
                box_buf.append(torch.tensor([[y0, x0, y1, x1]], dtype=torch.long))
                label_buf.append(torch.tensor([int(yb[i].item())], dtype=torch.long))
                score_buf.append(score_val.detach().cpu().view(1))
                quality_buf.append(torch.tensor([quality_flag * (rank + 1)], dtype=torch.long))
                buffered += 1

                if buffered >= crop_shard_size:
                    flush_buffers()

        del union, xb, yb, logits, probs_full, top1, target_idx, mean_union_map, frac_map, yh, xw, boxes, boxes_sorted, scores_sorted
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    flush_buffers()

    if total == 0:
        raise RuntimeError("No TGSC crops generated; verify teacher setup and TGSC thresholds.")

    manifest = {
        "num_samples": total,
        "resolution": [ch, cw],
        "format": "coords_v2",
        "shards": shards,
        "version": 2,
    }
    _write_json(manifest_path, manifest)
    print(f"[TGSC] Built disk crop cache for {ch}x{cw}: {total} samples, {len(shards)} shard(s).")
    return DiskCropTensorDataset(res_dir, base_dataset=base_dataset)


class TGSCPreprocessor:
    def __init__(self, teacher: nn.Module, data_cfg: DataConfig, tgsc_cfg: TGSCConfig, imagenet_ds: DatasetDict):
        self.teacher = teacher
        self.data_cfg = data_cfg
        self.tgsc_cfg = tgsc_cfg
        self.imagenet_ds = imagenet_ds
        self.cache_paths = ensure_cache_layout(data_cfg.cache_dir)
        self.target_layer = pick_target_layer(teacher)

        train_split = data_cfg.train_split if data_cfg.train_split in imagenet_ds else "train"
        if train_split not in imagenet_ds:
            raise RuntimeError(f"Train split '{data_cfg.train_split}' not found in dataset keys: {list(imagenet_ds.keys())}")

        self.base_transform = transforms.Compose([
            transforms.Resize(data_cfg.resize_size),
            transforms.CenterCrop(data_cfg.image_size),
            transforms.ToTensor(),
            transforms.Normalize(data_cfg.mean, data_cfg.std),
        ])
        self.base_dataset = ImageNetTorchDataset(imagenet_ds[train_split], transform=self.base_transform)

        self._cache: Dict[int, DiskCropTensorDataset] = {}
        self.last_dataset: Optional[DiskCropTensorDataset] = None
        self._teacher_cache: Optional[TeacherUnionCache] = None

    def _make_base_loader(self) -> DataLoader:
        return DataLoader(
            self.base_dataset,
            batch_size=self.tgsc_cfg.precompute_batch_size,
            shuffle=False,
            num_workers=self.tgsc_cfg.precompute_workers,
            pin_memory=self.data_cfg.pin_memory,
        )

    def _build_teacher_cache(self) -> TeacherUnionCache:
        base_loader = self._make_base_loader()
        return load_or_build_teacher_union_cache(
            base_loader,
            self.teacher,
            self.target_layer,
            cache_dir=self.cache_paths["teacher_union"],
            shard_size=self.tgsc_cfg.teacher_cache_shard_size,
            teacher_cache_batch_size=self.tgsc_cfg.teacher_cache_batch_size,
            teacher_cache_min_batch_size=self.tgsc_cfg.teacher_cache_min_batch_size,
            force_rebuild=self.tgsc_cfg.force_rebuild,
            use_gt_target=self.tgsc_cfg.use_gt_target,
            w_cam=self.tgsc_cfg.w_cam,
            w_sal=self.tgsc_cfg.w_sal,
            temp=self.tgsc_cfg.temp,
            gamma=self.tgsc_cfg.gamma,
            pct_lo=self.tgsc_cfg.pct_lo,
            pct_hi=self.tgsc_cfg.pct_hi,
        )

    def _ensure_dataset(self, crop_size: int) -> DiskCropTensorDataset:
        if self.tgsc_cfg.cache and crop_size in self._cache:
            print(f"[TGSC] Reusing in-session dataset handle for {crop_size}x{crop_size}.")
            return self._cache[crop_size]

        if self._teacher_cache is None:
            self._teacher_cache = self._build_teacher_cache()

        base_loader = self._make_base_loader()
        dataset = build_bestcrop_dataset_batchwise(
            base_loader,
            self.teacher,
            self.target_layer,
            crop_size=crop_size,
            cache_root=self.cache_paths["tgsc_crops"],
            base_dataset=self.base_dataset,
            crop_shard_size=self.tgsc_cfg.crop_cache_shard_size,
            force_rebuild=self.tgsc_cfg.force_rebuild,
            use_gt_target=self.tgsc_cfg.use_gt_target,
            w_cam=self.tgsc_cfg.w_cam,
            w_sal=self.tgsc_cfg.w_sal,
            temp=self.tgsc_cfg.temp,
            gamma=self.tgsc_cfg.gamma,
            pct_lo=self.tgsc_cfg.pct_lo,
            pct_hi=self.tgsc_cfg.pct_hi,
            stride=self.tgsc_cfg.stride,
            k_per_img=self.tgsc_cfg.k_per_img,
            min_frac=self.tgsc_cfg.min_frac,
            min_mean=self.tgsc_cfg.min_mean,
            eval_chunk=self.tgsc_cfg.eval_chunk,
            top_k_crops=self.tgsc_cfg.top_k_crops,
            low_k_crops=self.tgsc_cfg.low_k_crops,
            teacher_cache=self._teacher_cache,
        )

        if self.tgsc_cfg.cache:
            self._cache[crop_size] = dataset

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return dataset

    def get_loader(
        self,
        crop_size: int,
        *,
        batch_size: int,
        num_workers: int,
        persistent_workers: bool,
        pin_memory: bool,
    ) -> DataLoader:
        dataset = self._ensure_dataset(crop_size)

        aug_ops = []
        if self.tgsc_cfg.aug_padding > 0:
            aug_ops.append(
                transforms.RandomCrop(
                    crop_size,
                    padding=self.tgsc_cfg.aug_padding,
                    padding_mode="reflect",
                )
            )
        if self.tgsc_cfg.flip_prob > 0.0:
            aug_ops.append(transforms.RandomHorizontalFlip(p=self.tgsc_cfg.flip_prob))
        aug_tfm = transforms.Compose(aug_ops) if aug_ops else None

        dataset.set_transform(aug_tfm)
        self.last_dataset = dataset
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )


# --------------------------- Trainer -----------------------------------
class KDTrainer:
    def __init__(
        self,
        student: nn.Module,
        teacher: nn.Module,
        data_cfg: DataConfig,
        kd_cfg: KDConfig,
        opt_cfg: OptimConfig,
        train_cfg: TrainConfig,
        tgsc_cfg: TGSCConfig,
        *,
        imagenet_ds: Optional[DatasetDict] = None,
        device: Optional[torch.device] = None,
    ):
        self.student = student
        self.teacher = teacher
        self.data_cfg = data_cfg
        self.kd_cfg = kd_cfg
        self.opt_cfg = opt_cfg
        self.train_cfg = train_cfg
        self.tgsc_cfg = tgsc_cfg
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.cache_paths = ensure_cache_layout(data_cfg.cache_dir)
        self.imagenet_ds = imagenet_ds if imagenet_ds is not None else load_or_build_imagenet_ds(data_cfg)

        self.kd_loss = KDLoss(kd_cfg.T, kd_cfg.alpha)
        self.extra = ExtraLosses()

        _ = infer_num_classes(self.student, self.device)

        self.optimizer = optim.SGD(
            list(self.student.parameters()),
            lr=opt_cfg.base_lr,
            momentum=opt_cfg.momentum,
            weight_decay=opt_cfg.weight_decay,
        )

        if not opt_cfg.lr_half_on_stage_change:
            self.scheduler = optim.lr_scheduler.MultiStepLR(
                self.optimizer,
                milestones=list(opt_cfg.milestones),
                gamma=opt_cfg.milestone_gamma,
            )
        else:
            self.scheduler = None

        self.test_loader = make_val_loader(self.imagenet_ds, data_cfg)
        self.per_epoch_res = build_stage_plan(list(data_cfg.stages))
        self.tgsc_preproc: Optional[TGSCPreprocessor] = None
        self.current_dataset: Optional[DiskCropTensorDataset] = None

    def _prepare_models(self):
        self.teacher.to(self.device).eval()
        self.student.to(self.device).train()

    def _maybe_halve_lr(self, epoch_idx: int):
        if not self.opt_cfg.lr_half_on_stage_change or epoch_idx == 0:
            return
        prev_res = self.per_epoch_res[epoch_idx - 1]
        curr_res = self.per_epoch_res[epoch_idx]
        if curr_res != prev_res:
            for g in self.optimizer.param_groups:
                g["lr"] *= 0.5
            print(f"[LR] Stage change {prev_res}->{curr_res}: halved LR to {self.optimizer.param_groups[0]['lr']:.6f}")

    def _score_to_kd_weights(self, scores: torch.Tensor, quality_flags: torch.Tensor) -> torch.Tensor:
        del quality_flags
        cfg = self.tgsc_cfg
        eps = 1e-6
        weights = torch.ones_like(scores)
        mask = scores < cfg.score_high
        if cfg.score_high > eps:
            weights[mask] = scores[mask] / cfg.score_high
        return weights.clamp(min=0.0, max=1.5)

    def train(self):
        set_seed(self.train_cfg.seed)
        self._prepare_models()

        if self.tgsc_preproc is None:
            self.tgsc_preproc = TGSCPreprocessor(self.teacher, self.data_cfg, self.tgsc_cfg, self.imagenet_ds)

        total_epochs = len(self.per_epoch_res)
        assert total_epochs == self.train_cfg.epochs, (
            f"epochs ({self.train_cfg.epochs}) must equal sum of stage epochs ({total_epochs})"
        )

        best_test = -1.0
        current_res = None
        accum_steps = max(1, int(self.train_cfg.grad_accum_steps))

        for epoch in range(total_epochs):
            res = self.per_epoch_res[epoch]
            stage_changed = res != current_res
            if stage_changed:
                current_res = res
                self._maybe_halve_lr(epoch)

            train_loader = self.tgsc_preproc.get_loader(
                crop_size=res,
                batch_size=self.data_cfg.batch_size,
                num_workers=self.data_cfg.num_workers,
                persistent_workers=self.data_cfg.persistent_workers,
                pin_memory=self.data_cfg.pin_memory,
            )
            self.current_dataset = self.tgsc_preproc.last_dataset

            running = {"loss": 0.0, "ce": 0.0, "kd": 0.0, "n": 0, "correct": 0}
            self.optimizer.zero_grad(set_to_none=True)

            it = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{total_epochs} | res={res}", leave=False)
            for step, (x, y, scores, qflags, _idx) in enumerate(it):
                x = x.to(self.device, non_blocking=True)
                y = y.to(self.device, non_blocking=True)
                score_t = scores.to(self.device, non_blocking=True).float()
                qflag_t = qflags.to(self.device, non_blocking=True).float()

                with torch.no_grad():
                    t_logits = extract_logits(self.teacher(x))

                s_logits = extract_logits(self.student(x))
                kd_weights = self._score_to_kd_weights(score_t, qflag_t)
                kd_out = self.kd_loss(s_logits, t_logits, y, weights=kd_weights)

                extra = self.extra.forward(device=self.device)
                loss = kd_out["loss"] + extra["extra_loss"]

                (loss / accum_steps).backward()

                should_step = ((step + 1) % accum_steps == 0) or ((step + 1) == len(train_loader))
                if should_step:
                    self.optimizer.step()
                    self.optimizer.zero_grad(set_to_none=True)

                with torch.no_grad():
                    pred = s_logits.argmax(dim=1)
                    correct = (pred == y).sum().item()

                bs = y.size(0)
                running["loss"] += float(loss.item()) * bs
                running["ce"] += float(kd_out["ce"].item()) * bs
                running["kd"] += float(kd_out["kd"].item()) * bs
                running["n"] += bs
                running["correct"] += correct

            if self.scheduler is not None:
                self.scheduler.step()

            train_loss = running["loss"] / max(1, running["n"])
            train_ce = running["ce"] / max(1, running["n"])
            train_kd = running["kd"] / max(1, running["n"])
            train_acc = 100.0 * running["correct"] / max(1, running["n"])
            test_acc = evaluate(self.student, self.test_loader, self.device)

            lr_now = self.optimizer.param_groups[0]["lr"]
            print(
                f"Epoch {epoch + 1:3d}/{total_epochs} | res {res:3d} | "
                f"loss {train_loss:.4f} (CE {train_ce:.4f}, KD {train_kd:.4f}) | "
                f"train {train_acc:5.2f}% | val {test_acc:5.2f}% | lr {lr_now:.6f}"
            )

            if test_acc > best_test:
                best_test = test_acc
                sd = self.student.module.state_dict() if isinstance(self.student, nn.DataParallel) else self.student.state_dict()
                save_path = Path(self.train_cfg.save_path)
                save_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(sd, str(save_path))
                print(f"  -> Saved best checkpoint @ epoch {epoch + 1} to {save_path}")

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"Finished. Best Val Acc: {best_test:.2f}%")


# MAIN


In [ ]:
# ----------------------------- Main ------------------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    import atexit, sys, os
    @atexit.register
    def _squelch_multiproc_exit_noise():
        try:
            sys.stderr = open(os.devnull, "w")
        except Exception:
            pass

    

    resolved_root = str(resolve_imagenet_root("./data"))
    data_cfg = DataConfig(
        root=resolved_root,
        cache_dir="./.cache",
        stages=((96, 20), (128, 20), (160, 20), (192, 20), (224, 20)),
        batch_size=256,
        eval_batch_size=128,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        image_size=224,
        resize_size=256,
    )

    cache_paths = ensure_cache_layout(data_cfg.cache_dir)
    imagenet_ds = load_or_build_imagenet_ds(data_cfg)

    teacher = resnet34(
        pretrained=True,
        checkpoint_dir=str(cache_paths["checkpoints"]),
        num_classes=1000,
    )
    student = resnet18(
        pretrained=False,
        checkpoint_dir=str(cache_paths["checkpoints"]),
        num_classes=1000,
    )

    kd_cfg = KDConfig(T=1.0, alpha=0.9)
    opt_cfg = OptimConfig(
        base_lr=0.1,
        momentum=0.9,
        weight_decay=1e-4,
        lr_half_on_stage_change=True,
        milestones=(30, 60, 90),
        milestone_gamma=0.1,
    )
    train_cfg = TrainConfig(
        epochs=100,
        grad_accum_steps=1,
        seed=27,
        save_path=str(cache_paths["checkpoints"] / "student_r18_kd_imagenet.pth"),
    )
    tgsc_cfg = TGSCConfig(
        precompute_batch_size=128,
        precompute_workers=8,
        use_gt_target=True,
        w_cam=0.7,
        w_sal=0.3,
        temp=1.5,
        gamma=1.2,
        pct_lo=1.0,
        pct_hi=99.0,
        stride=8,
        k_per_img=6,
        min_frac=0.20,
        min_mean=0.20,
        eval_chunk=128,
        cache=True,
        force_rebuild=False,
        flip_prob=0.5,
        aug_padding=2,
        top_k_crops=2,
        low_k_crops=1,
        score_high=0.75,
        score_low=0.35,
        teacher_cache_shard_size=128,
        teacher_cache_batch_size=64,
        teacher_cache_min_batch_size=8,
        crop_cache_shard_size=4096,
    )

    print("[RUN] Effective batch size:", data_cfg.batch_size * train_cfg.grad_accum_steps)
    print("[RUN] Stage schedule:", list(data_cfg.stages))
    print("[RUN] LR policy: halve on each stage transition")

    trainer = KDTrainer(
        student,
        teacher,
        data_cfg,
        kd_cfg,
        opt_cfg,
        train_cfg,
        tgsc_cfg
        ,
        imagenet_ds=imagenet_ds,
        device=device,
    )

    _squelch_multiproc_exit_noise()
    trainer.train()


[DATA] Loading cached ImageNet DatasetDict from .cache/imagenet_ds/saved_ds


Loading dataset from disk:   0%|          | 0/294 [00:00<?, ?it/s]

[RUN] Effective batch size: 256
[RUN] Stage schedule: [(96, 20), (128, 20), (160, 20), (192, 20), (224, 20)]
[RUN] LR policy: halve on each stage transition
[TGSC] Reusing teacher-union cache from .cache/teacher_union
[TGSC] Reusing disk crop cache for 96x96 at .cache/tgsc_crops/res_96x96


Epoch 1/100 | res=96:   0%|          | 0/15014 [00:00<?, ?it/s]

Epoch   1/100 | res  96 | loss 2.7622 (CE 3.9216, KD 2.6334) | train 24.67% | val 32.37% | lr 0.100000
  -> Saved best checkpoint @ epoch 1 to .cache/checkpoints/student_r18_kd_imagenet.pth
[TGSC] Reusing in-session dataset handle for 96x96.


...

Epoch 22/100 | res=128:   0%|          | 0/15014 [00:00<?, ?it/s]

Epoch  22/100 | res 128 | loss 1.0295 (CE 1.8001, KD 0.9439) | train 59.03% | val 54.55% | lr 0.050000
  -> Saved best checkpoint @ epoch 22 to .cache/checkpoints/student_r18_kd_imagenet.pth
[TGSC] Reusing in-session dataset handle for 128x128.


Epoch 23/100 | res=128:   0%|          | 0/15014 [00:00<?, ?it/s]

Epoch  23/100 | res 128 | loss 1.0188 (CE 1.7873, KD 0.9334) | train 59.32% | val 54.95% | lr 0.050000
  -> Saved best checkpoint @ epoch 23 to .cache/checkpoints/student_r18_kd_imagenet.pth
[TGSC] Reusing in-session dataset handle for 128x128.


Epoch 24/100 | res=128:   0%|          | 0/15014 [00:00<?, ?it/s]

In [ ]:
# import torch
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# import numpy as np
# from pathlib import Path
# from typing import Optional, Tuple, List
# from torchvision import datasets, transforms


# class TGSCCropVisualizer:
#     """
#     Visualize TGSC crops: the crop itself, original image with bbox, and union map overlay.
#     Shows 3 good crops (high scores) and 3 bad crops (low scores) per resolution.
#     """
    
#     def __init__(
#         self,
#         dataset_root: str = "./data",
#         mean: Tuple[float, float, float] = (0.5071, 0.4867, 0.4408),
#         std: Tuple[float, float, float] = (0.2675, 0.2565, 0.2761),
#         save_dir: str = "./crop_visualizations",
#     ):
#         self.dataset_root = dataset_root
#         self.mean = torch.tensor(mean).view(3, 1, 1)
#         self.std = torch.tensor(std).view(3, 1, 1)
#         self.save_dir = Path(save_dir)
#         self.save_dir.mkdir(parents=True, exist_ok=True)
        
#         # Load CIFAR-100 dataset (raw images, no normalization)
#         self.cifar_dataset = datasets.CIFAR100(
#             root=dataset_root, 
#             train=True, 
#             download=True, 
#             transform=transforms.ToTensor()
#         )
        
#         self.class_names = self.cifar_dataset.classes
    
#     def denormalize(self, tensor: torch.Tensor) -> torch.Tensor:
#         """Denormalize a normalized tensor back to [0, 1] range."""
#         return tensor * self.std + self.mean
    
#     def tensor_to_image(self, tensor: torch.Tensor) -> np.ndarray:
#         """Convert tensor to numpy array for visualization."""
#         img = self.denormalize(tensor).clamp(0, 1)
#         return img.permute(1, 2, 0).cpu().numpy()
    
#     def visualize_single_crop(
#         self,
#         crop_tensor: torch.Tensor,
#         source_idx: int,
#         crop_bbox: Tuple[int, int, int, int],  # (y0, x0, y1, x1)
#         union_map: Optional[torch.Tensor],
#         label: int,
#         score: float,
#         quality_flag: int,
#         save_path: Optional[Path] = None,
#     ):
#         """
#         Visualize a single crop with its original image and union map.
        
#         Args:
#             crop_tensor: [3, H, W] normalized crop
#             source_idx: Index of source image in CIFAR-100
#             crop_bbox: Bounding box coordinates (y0, x0, y1, x1)
#             union_map: [1, 32, 32] attention union map
#             label: Class label
#             score: Quality score from teacher
#             quality_flag: +1 (good), -1 (bad)
#             save_path: Optional path to save figure
#         """
#         fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
#         # 1. The Crop Itself
#         crop_img = self.tensor_to_image(crop_tensor)
#         axes[0].imshow(crop_img)
#         axes[0].set_title(
#             f"Crop ({crop_tensor.shape[1]}×{crop_tensor.shape[2]})\n"
#             f"Score: {score:.3f} | Quality: {'Good' if quality_flag > 0 else 'Bad'}"
#         )
#         axes[0].axis('off')
        
#         # 2. Original Image with Bounding Box
#         orig_img, _ = self.cifar_dataset[source_idx]  # [3, 32, 32]
#         orig_img_np = orig_img.permute(1, 2, 0).numpy()
        
#         axes[1].imshow(orig_img_np)
        
#         # Draw bounding box
#         y0, x0, y1, x1 = crop_bbox
#         rect = patches.Rectangle(
#             (x0, y0), x1 - x0, y1 - y0,
#             linewidth=2,
#             edgecolor='red' if quality_flag > 0 else 'blue',
#             facecolor='none'
#         )
#         axes[1].add_patch(rect)
#         axes[1].set_title(
#             f"Original Image (32×32)\n"
#             f"Class: {self.class_names[label]} | Source: {source_idx}"
#         )
#         axes[1].axis('off')j
        
#         # 3. Union Map Overlay
#         axes[2].imshow(orig_img_np)
        
#         if union_map is not None:
#             # Overlay union map as heatmap
#             union_np = union_map.squeeze(0).cpu().numpy()
#             im = axes[2].imshow(
#                 union_np,
#                 alpha=0.6,
#                 cmap='jet',
#                 interpolation='bilinear'
#             )
#             plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
        
#         # Draw bounding box on overlay too
#         rect2 = patches.Rectangle(
#             (x0, y0), x1 - x0, y1 - y0,
#             linewidth=2,
#             edgecolor='lime',
#             facecolor='none'
#         )
#         axes[2].add_patch(rect2)
#         axes[2].set_title("Union Map Overlay\n(Teacher Attention)")
#         axes[2].axis('off')
        
#         plt.tight_layout()
        
#         if save_path:
#             plt.savefig(save_path, dpi=150, bbox_inches='tight')
#             print(f"Saved: {save_path}")
        
#         return fig
    
#     def visualize_dataset_samples(
#         self,
#         crop_dataset,  # CropTensorDataset instance
#         union_cache,  # TeacherUnionCache instance
#         crop_size: int,
#         num_good: int = 3,
#         num_bad: int = 3,
#     ):
#         """
#         Visualize best and worst crops from a dataset.
        
#         Args:
#             crop_dataset: CropTensorDataset instance
#             union_cache: TeacherUnionCache with union maps
#             crop_size: Current crop resolution
#             num_good: Number of good (high-score) crops to visualize
#             num_bad: Number of bad (low-score) crops to visualize
#         """
#         # Get scores and sort
#         scores = crop_dataset.scores.numpy()
#         sorted_indices = np.argsort(scores)
        
#         # Select top and bottom samples
#         good_indices = sorted_indices[-num_good:][::-1]  # Highest scores
#         bad_indices = sorted_indices[:num_bad]  # Lowest scores
        
#         # Create subdirectory for this resolution
#         res_dir = self.save_dir / f"resolution_{crop_size}x{crop_size}"
#         res_dir.mkdir(exist_ok=True)
        
#         print(f"\n{'='*60}")
#         print(f"Visualizing crops for resolution {crop_size}×{crop_size}")
#         print(f"{'='*60}\n")
        
#         # Visualize good crops
#         print(f"📊 Good Crops (Top {num_good} scores):")
#         for rank, idx in enumerate(good_indices, 1):
#             crop, label, score, qflag, _ = crop_dataset[idx]
#             source_idx = int(crop_dataset.source_indices[idx].item())
            
#             # Reconstruct bounding box (approximate from crop size)
#             # Note: We don't have exact bbox stored, so we'll use a placeholder
#             # In practice, you'd need to store bboxes in your dataset
#             bbox = self._reconstruct_bbox(crop, source_idx, crop_size)
            
#             # Get union map for this source
#             union_map = union_cache.union_maps[source_idx]
            
#             save_path = res_dir / f"good_rank{rank}_score{score:.3f}_src{source_idx}.png"
            
#             print(f"  Rank {rank}: Score={score:.3f}, Label={self.class_names[label]}, Source={source_idx}")
            
#             fig = self.visualize_single_crop(
#                 crop_tensor=crop,
#                 source_idx=source_idx,
#                 crop_bbox=bbox,
#                 union_map=union_map,
#                 label=label,
#                 score=score,
#                 quality_flag=qflag,
#                 save_path=save_path,
#             )
#             plt.show()
        
#         # Visualize bad crops
#         print(f"\n📊 Bad Crops (Bottom {num_bad} scores):")
#         for rank, idx in enumerate(bad_indices, 1):
#             crop, label, score, qflag, _ = crop_dataset[idx]
#             source_idx = int(crop_dataset.source_indices[idx].item())
            
#             bbox = self._reconstruct_bbox(crop, source_idx, crop_size)
#             union_map = union_cache.union_maps[source_idx]
            
#             save_path = res_dir / f"bad_rank{rank}_score{score:.3f}_src{source_idx}.png"
            
#             print(f"  Rank {rank}: Score={score:.3f}, Label={self.class_names[label]}, Source={source_idx}")
            
#             fig = self.visualize_single_crop(
#                 crop_tensor=crop,
#                 source_idx=source_idx,
#                 crop_bbox=bbox,
#                 union_map=union_map,
#                 label=label,
#                 score=score,
#                 quality_flag=qflag,
#                 save_path=save_path,
#             )
#             plt.close(fig)
        
#         print(f"\n✅ All visualizations saved to: {res_dir}")
    
#     def _reconstruct_bbox(
#         self, 
#         crop: torch.Tensor, 
#         source_idx: int, 
#         crop_size: int
#     ) -> Tuple[int, int, int, int]:
#         """
#         Placeholder for bbox reconstruction.
#         In practice, you should store bboxes in your CropTensorDataset.
#         For now, we'll just center the crop.
#         """
#         orig_h, orig_w = 32, 32
#         crop_h, crop_w = crop.shape[1], crop.shape[2]
        
#         # Center the bbox (rough approximation)
#         y0 = (orig_h - crop_h) // 2
#         x0 = (orig_w - crop_w) // 2
#         y1 = y0 + crop_h
#         x1 = x0 + crop_w
        
#         return (y0, x0, y1, x1)
    
#     def visualize_all_resolutions(
#         self,
#         tgsc_preprocessor,  # TGSCPreprocessor instance with cached datasets
#         resolutions: List[int] = [16, 20, 24, 28, 32],
#         num_good: int = 3,
#         num_bad: int = 3,
#     ):
#         """
#         Visualize crops for all resolutions.
        
#         Args:
#             tgsc_preprocessor: TGSCPreprocessor instance with built caches
#             resolutions: List of crop resolutions to visualize
#             num_good: Number of good crops per resolution
#             num_bad: Number of bad crops per resolution
#         """
#         if tgsc_preprocessor._teacher_cache is None:
#             raise ValueError(
#                 "Teacher cache not built! Run training first or call "
#                 "tgsc_preprocessor._build_teacher_cache()"
#             )
        
#         union_cache = tgsc_preprocessor._teacher_cache
        
#         for res in resolutions:
#             if res not in tgsc_preprocessor._cache:
#                 print(f"⚠️  No cached dataset for resolution {res}×{res}, skipping...")
#                 continue
            
#             crop_dataset = tgsc_preprocessor._cache[res]
            
#             self.visualize_dataset_samples(
#                 crop_dataset=crop_dataset,
#                 union_cache=union_cache,
#                 crop_size=res,
#                 num_good=num_good,
#                 num_bad=num_bad,
#             )


# # ==================== STANDALONE USAGE EXAMPLE ====================

# def visualize_before_training(
#     teacher_path: str,
#     resolutions: List[int] = [16, 20, 24, 28, 32],
#     num_good: int = 3,
#     num_bad: int = 3,
#     save_dir: str = "./crop_visualizations_pretrain",
# ):
#     """
#     Generate and visualize crops BEFORE training starts.
#     This helps you inspect what the cropping algorithm selects.
    
#     Args:
#         teacher_path: Path to teacher checkpoint
#         resolutions: List of crop sizes to visualize
#         num_good: Number of good crops per resolution
#         num_bad: Number of bad crops per resolution
#         save_dir: Where to save visualizations
#     """
#     print("="*60)
#     print("TGSC Crop Visualizer - Pre-Training Inspection")
#     print("="*60)
    
#     # You need to import your training modules
#     # Uncomment and adjust the import path:
#     # from kd_crop_curriculum_refactor import (
#     #     TGSCPreprocessor, DataConfig, TGSCConfig,
#     #     resnet110, torch
#     # )
    
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
#     print(f"\n1️⃣  Loading teacher model from: {teacher_path}")
#     teacher = resnet110(num_classes=100)
    
#     # Load checkpoint
#     ckpt = torch.load(teacher_path, map_location=device, weights_only=False)
#     if "model" in ckpt:
#         teacher.load_state_dict(ckpt["model"])
#     else:
#         teacher.load_state_dict(ckpt)
    
#     teacher.to(device).eval()
#     print(f"✅ Teacher loaded on {device}")
    
#     print("\n2️⃣  Setting up TGSC preprocessor...")
#     data_cfg = DataConfig(root="./data")
#     tgsc_cfg = TGSCConfig(cache=True)
    
#     tgsc_preproc = TGSCPreprocessor(teacher, data_cfg, tgsc_cfg)
    
#     print("\n3️⃣  Generating crops for all resolutions...")
#     print("   (This may take a few minutes...)")

#     print("\n4️⃣  Creating visualizations...")
#     visualizer = TGSCCropVisualizer(
#     dataset_root="./data",
#     save_dir=save_dir
#     )

#     for res in resolutions:
#         print(f"   📐 Building crops for {res}×{res}...")
#         _ = tgsc_preproc.get_dataset(res)
#         visualizer.visualize_dataset_samples(
#         crop_dataset=tgsc_preproc._cache[res],
#         union_cache=tgsc_preproc._teacher_cache,
#         crop_size=res,
#         num_good=num_good,
#         num_bad=num_bad,
#         )
    
#     print(f"\n✅ Visualization complete! Check {save_dir}/")
#     print("\nYou can now:")
#     print("  - Inspect the crop quality")
#     print("  - Verify attention maps make sense")
#     print("  - Tune TGSC hyperparameters if needed")
#     print("  - Then proceed with training")


# if __name__ == "__main__":
#     """
#     Run this script to visualize crops BEFORE starting training.
    
#     Usage:
#         python tgsc_visualizer.py
    
#     Or with custom settings:
#         python -c "from tgsc_visualizer import visualize_before_training; \
#                    visualize_before_training('path/to/teacher.pth')"
#     """
#     import sys
    
#     # Default teacher path - CHANGE THIS to your teacher checkpoint
#     teacher_checkpoint = "/kaggle/input/resnet110/pytorch/default/1/ckpt_epoch_240.pth"
    
#     # Check if path exists
#     if not Path(teacher_checkpoint).exists():
#         print(f"❌ Error: Teacher checkpoint not found at {teacher_checkpoint}")
#         print("\nPlease update the teacher_checkpoint path in this script.")
#         sys.exit(1)
    
#     # Run visualization
#     visualize_before_training(
#         teacher_path=teacher_checkpoint,
#         resolutions=[16, 20, 24, 28, 32],
#         num_good=3,
#         num_bad=3,
#         save_dir="./crop_visualizations_pretrain"
#     )
    
#     print("\n" + "="*60)
#     print("Next steps:")
#     print("  1. Review visualizations in ./crop_visualizations_pretrain/")
#     print("  2. Adjust TGSCConfig hyperparameters if needed")
#     print("  3. Run training with: python kd_crop_curriculum_refactor.py")
#     print("="*60)

In [ ]:
# # After visualizer.visualize_dataset_samples():
# from IPython.display import Image, display
# save_dir="./crop_visualizations_pretrain"
# res_list = [16,20,24,28,32]
# for res in res_list:
#     res_dir = Path(save_dir) / f"resolution_{res}x{res}"
#     print(f"\n📊 Displaying visualizations for {res}×{res}:")
    
#     for img_path in sorted(res_dir.glob("*.png")):
#         print(f"\n{img_path.name}")
#         display(Image(filename=str(img_path)))